# Does it generalize beyond onion? Cross-source potato disease diagnosis
### Second benchmark for the SERA pipeline: PlantVillage · PlantDoc · PLD (Pakistan) · Uncontrolled Environment (Indonesia)

**Why this notebook.** The onion study (v2) found three things: (1) public sources are separable by *acquisition* alone (metadata → source 0.91 balanced accuracy; within a disease 0.93–0.96), and dataset style is a stronger signal than the disease; (2) calibration on the training sources' validation data exceeds a 5% wrong-treatment target ~5× on unseen sources; (3) leave-one-source-out (LOSO) conformal risk control with a kNN distance score meets it. SERA itself gave no significant gain. A Q1 reviewer will ask whether this is onion-specific. This notebook runs the **identical pipeline, unchanged hyper-parameters** on a second crop, using public data only.

**Why potato, and why these four sources.** The literature's standard shift, PlantVillage → PlantDoc (e.g. Xiang et al., *Front. Plant Sci.* 2026), has only **one** training source, so leave-one-source-out calibration is impossible there. Potato is the one crop where four *distinct* public sources share the same diseases:

| source | Kaggle dataset (attach) | origin | conditions | classes used |
|---|---|---|---|---|
| `plantvillage` | `abdallahalidev/plantvillage-dataset` (folder `color`) | USA (Hughes & Salathé 2015) | lab, uniform background | early blight, late blight, healthy |
| `plantdoc` | `nirmalsankalana/plantdoc-dataset` (train + test) | web-scraped (Singh et al., CoDS-COMAD 2020) | field / internet photos | early blight, late blight |
| `pld_pakistan` | `rizwan123456789/potato-disease-leaf-datasetpld` | Central Punjab, Pakistan (Rashid et al., *Electronics* 2021) | field, leaves cropped by YOLOv5 | early blight, late blight, healthy |
| `uncontrolled_indonesia` | `warcoder/potato-leaf-disease-dataset` | Shabrina et al. 2023, Mendeley Data, doi 10.17632/ptz377bwb8.1 | uncontrolled field backgrounds | Phytophthora (= late blight), healthy, + fungi, bacteria, virus, pest, nematode |

**Excluded on purpose (provenance check):** `nafishamoin/new-bangladeshi-crop-disease` has potato folders with exactly 1000 / 152 / 1000 images, identical to PlantVillage's potato counts, so it is very likely a PlantVillage copy rather than Bangladeshi data. `muhammadardiputra/potato-leaf-disease-dataset` has unknown origin and more healthy images (500) than PlantVillage has, which suggests augmentation. The dedup step (exact + perceptual hash across sources) runs as in onion and reports any remaining cross-source copies.

**Label harmonization.** Same disease, same name: `early_blight`, `late_blight` (Indonesia's *Phytopthora*), `healthy`. Indonesia's other classes stay separate (`fungi_other`, `bacterial`, `viral`, `pest`, `nematode`). They exist in only one source, so when Indonesia is held out they are **unseen diseases**, a direct test of whether the gate withholds advice it cannot give. `fungi_other` shares the fungal-foliar remedy group with early blight, so issuing that group for it counts as correct.

**Protocols.** LODO over the four sources (inner leave-one-source-out runs for gate calibration), in-domain group 5-fold CV, plus the classic **single-source reference PlantVillage → PlantDoc** (only val-CRC is possible there; the comparison shows exactly why multi-source calibration matters).

**Pre-registered replication criteria** (same rules as onion; nothing is tuned on held-out sources):
- **R1 shortcut**: source predictable well above chance from metadata alone and from DINOv3 features within early blight / late blight / healthy.
- **R2 labels**: label-consistency index of `early_blight`, `late_blight`, `healthy` across sources (onion: consistent after source-centring).
- **R3 calibration**: val-CRC exceeds ε = 5% wrong-treatment risk on held-out sources, while LOSO-CRC (Remedy+kNN) meets it in most runs.
- **R4 explanations**: removing the top-attended view lowers the prediction more than removing a random view.
- SERA vs plain probe vs fine-tuned ViT-S/ConvNeXt-T/EfficientNet-B0: reported whatever the outcome.

### How to run
1. Settings: **GPU T4 x2** (or P100), **Internet ON**.
2. **Add Input → Datasets**, search and add all four: `abdallahalidev/plantvillage-dataset`, `nirmalsankalana/plantdoc-dataset`, `rizwan123456789/potato-disease-leaf-datasetpld`, `warcoder/potato-leaf-disease-dataset`.
3. Optional `SMOKE = True` → Run All (~20 min). Then `SMOKE = False` → **Save Version → Save & Run All (Commit)**. Expected ≈ 8–9 GPU-h (features ≈ 0.7 h, heads ≈ 0.7 h, 48 fine-tuning runs ≈ 6.5 h; time-guarded). To resume, attach this notebook's own output and run again.

Sections 3–17 are the same code as the onion notebook (v2.1). Only the data section (2) and the label map differ.

## 0. Configuration

In [ ]:
# ============================== CONFIG (potato) ==============================
SMOKE = False               # True = tiny subset: checks the whole notebook in ~20 min
MERGE_VIRAL = False         # (onion-only option; kept for the shared code)
IMG_SIZE = 224
CACHE_SIZE = 256
EPOCHS = 15                 # fine-tuned baselines: same recipe as onion v1
BATCH = 64
SEEDS = [0, 1, 2]
CV_FOLDS = 5
VAL_FOLDS = 7
DEDUP_HAMMING = 4
MIN_CLASS_IMAGES = 20
ALPHA = 0.10
TIME_BUDGET_H = 11.0
NUM_WORKERS = 8
DATASETS = ["plantvillage", "plantdoc", "pld_pakistan", "uncontrolled_indonesia"]
REGION = {"plantvillage": "USA (lab)", "plantdoc": "web (field photos)", "pld_pakistan": "Pakistan",
          "uncontrolled_indonesia": "Indonesia"}
EXTRA_SPLITS = [("plantvillage>plantdoc", ["plantvillage"], ["plantdoc"])]   # classic single-source reference

# ---- frozen foundation encoders, multi-scale tiles, SERA heads (identical to onion v2) ----
FM_ENCODERS = {
    "DINOv3-B":  dict(kind="timm", name="vit_base_patch16_dinov3.lvd1689m"),
    "DINOv3-S":  dict(kind="timm", name="vit_small_patch16_dinov3.lvd1689m"),
    "BioCLIP-2": dict(kind="open_clip", name="hf-hub:imageomics/bioclip-2"),
}
MAIN_FM, EDGE_FM = "DINOv3-B", "DINOv3-S"
TILE_GRIDS = (1, 2, 3)
TILE_CANVAS = 672
HEAD_SEEDS = list(range(10))
INNER_SEEDS = [0, 1, 2]
HEAD_EPOCHS = 40
HEAD_BATCH = 256
TILE_DROP = 0.25
KNN_K = 10
RISK_EPS = 0.05
MATCHED_ACCEPT = (0.3, 0.5)
FT_MODELS = ["ConvNeXtT", "ViTS", "EfficientNetB0"]
FT_GATE_MODELS = ["ViTS"]   # trained first on LODO + all leave-one-source-out splits (with kNN distances) for the gate
XAI_SAMPLES = 64
AUDIT_PER_CELL = 20
EXTERNAL_NAME = "potato_external"

CROP = "potato"
REMEDY_GROUP = {   # diseases that share the same management action
    "healthy": "none", "early_blight": "fungal_foliar", "fungi_other": "fungal_foliar",
    "late_blight": "oomycete", "bacterial": "bacterial", "viral": "vector_control",
    "pest": "pest_control", "nematode": "nematode_control",
}
ACTION = {   # advice templates (UNVERIFIED; agronomists must check them before any field use)
    "none": "No disease detected. Keep routine scouting and balanced fertilization.",
    "fungal_foliar": "Early blight / fungal leaf spot: remove infected lower leaves, avoid overhead irrigation, balanced nitrogen, rotate crops; protectant fungicide per local guideline.",
    "oomycete": "Late blight (urgent, spreads fast): remove and destroy infected plants, avoid leaf wetness, use certified seed tubers; late-blight fungicide programme per local guideline; warn neighbouring growers.",
    "bacterial": "Bacterial disease: use certified seed, avoid working in wet fields, remove infected plants; copper products only where the local guideline allows.",
    "vector_control": "Viral disease: no cure. Control aphid/whitefly vectors, rogue infected plants, use certified virus-free seed.",
    "pest_control": "Insect/mite damage: confirm the pest, apply threshold-based IPM (scouting, natural enemies, selective insecticide only if needed).",
    "nematode_control": "Nematodes: confirm with a soil test; rotate with non-host crops, use resistant varieties and clean seed.",
}

import os
from pathlib import Path
RUN_FT_BASELINES = os.environ.get("ONION_SKIP_FT", "0") != "1"
OUT = Path(os.environ.get("ONION_OUT", "/kaggle/working"))
INPUT_ROOT = Path(os.environ.get("ONION_INPUT", "/kaggle/input"))
CACHE_DIR = Path(os.environ.get("ONION_CACHE", "/tmp/potato_cache"))
PRETRAINED = os.environ.get("ONION_PRETRAINED", "1") == "1"
RES = OUT / "results"
RUNS, WEIGHTS, FIGS, TABLES = RES / "runs", RES / "weights", RES / "figures", RES / "tables"
FEATS = RES / "features"
HEADS = RES / "heads"
for p in (RUNS, WEIGHTS, FIGS, TABLES, FEATS, HEADS, CACHE_DIR):
    p.mkdir(parents=True, exist_ok=True)

if SMOKE:
    EPOCHS, SEEDS, CV_FOLDS, XAI_SAMPLES, AUDIT_PER_CELL = 1, [0], 2, 8, 3
    HEAD_SEEDS, INNER_SEEDS, HEAD_EPOCHS = [0, 1], [0], 6
    SMOKE_PER_CLASS = 24
    if os.environ.get("ONION_SMOKE_FMS"):
        FM_ENCODERS = {k: v for k, v in FM_ENCODERS.items() if k in os.environ["ONION_SMOKE_FMS"].split(",")}
print("SMOKE" if SMOKE else "FULL", "run | crop:", CROP, "| output ->", RES, "| encoders:", list(FM_ENCODERS))

## 0b. Package check (runs before any import)
Kaggle's preinstalled `timm` may be too old for DINOv3 (needs ≥ 1.0.20), and `open_clip` (BioCLIP 2) is not preinstalled. This cell upgrades only what is missing (needs Internet ON).

In [ ]:
import importlib.metadata as _md, subprocess, sys
def _ver(pkg):
    try:
        return tuple(int(x) for x in _md.version(pkg).split(".")[:3] if x.isdigit())
    except Exception:
        return None
_need = []
if (_ver("timm") or (0,)) < (1, 0, 20):
    _need.append("timm>=1.0.20")          # DINOv3 needs timm >= 1.0.20
if any(v["kind"] == "open_clip" for v in FM_ENCODERS.values()) and _ver("open_clip_torch") is None:
    _need.append("open_clip_torch")
if _ver("onnxscript") is None:
    _need.append("onnxscript")            # ONNX export of DINOv3 (torch.export path)
if _need:
    print("installing:", _need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *_need], check=False)
print("timm", _md.version("timm") if _ver("timm") else "missing",
      "| open_clip", _md.version("open_clip_torch") if _ver("open_clip_torch") else "missing")

## 1. Imports and utilities

In [ ]:
import re, io, json, math, time, random, shutil, hashlib, zipfile, warnings
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy import stats
from scipy.fft import dct
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
T0 = time.time()
print("torch", torch.__version__, "| timm", timm.__version__, "| device", DEVICE,
      torch.cuda.get_device_name(0) if AMP else "")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def hours():
    return (time.time() - T0) / 3600

def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))

def md_table(df, floatfmt="{:.3f}"):
    """DataFrame -> GitHub markdown (no extra dependency)."""
    df = df.reset_index() if not isinstance(df.index, pd.RangeIndex) else df
    cols = [str(c) for c in df.columns]
    fmt = lambda v: floatfmt.format(v) if isinstance(v, (float, np.floating)) and not pd.isna(v) else str(v)
    rows = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    rows += ["| " + " | ".join(fmt(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(rows)

REPORT = []   # (title, markdown) pairs collected for REPORT.md
try:
    from IPython.display import display, Markdown
    IN_NOTEBOOK = get_ipython() is not None
except Exception:
    IN_NOTEBOOK = False

def report(title, text):
    """Show a result block nicely in the notebook and keep it for REPORT.md."""
    REPORT.append((title, text))
    if IN_NOTEBOOK:
        display(Markdown(f"#### {title}\n\n{text}"))
    else:
        print(f"\n## {title}\n{text}")

## 1b. Resume: pick up runs from earlier versions of this notebook
Any attached input that contains `results/runs/*.json` (an earlier output of this notebook) is copied in, so finished runs are skipped.

In [ ]:
def resume_previous():
    """Copy finished runs (v1 fine-tuned baselines, v2 heads) and cached tile features from attached outputs.
    v1 model weights are NOT copied (not needed in v2; saves several GB)."""
    n = 0
    if not INPUT_ROOT.exists():
        return 0
    for runs_dir in INPUT_ROOT.rglob("results/runs"):
        if runs_dir.resolve() == RUNS.resolve():
            continue
        for f in runs_dir.iterdir():
            if not (RUNS / f.name).exists():
                shutil.copy2(f, RUNS / f.name); n += 1
        for sub, dst in (("features", FEATS), ("heads", HEADS)):
            sdir = runs_dir.parent / sub
            if sdir.exists():
                for f in sdir.iterdir():
                    if sub == "features" and not f.name.startswith(("tiles_", "acqmeta_")):
                        continue                       # v1 global features are not used
                    if not (dst / f.name).exists():
                        shutil.copy2(f, dst / f.name); n += 1
        print("resumed from", runs_dir.parent)
    return n

print("files resumed:", resume_previous())

## 2. Get the data: four attached Kaggle datasets → one folder per source
Each source's potato class folders are located anywhere under `/kaggle/input` (by folder name + a source-specific path rule), and copied into `/tmp/potato_cache/data/<source>/<class>/`. Files keep their original name, prefixed with their original split folder so names cannot collide. The train/validation/test splits of the original datasets are **ignored**: all splits below are made by this notebook (group-aware, leakage-checked).

In [ ]:
import socket
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
DATA_LOCAL = CACHE_DIR / "data"

POTATO_SOURCES = {   # source -> (Kaggle dataset to attach, class folder names, rule that picks the right folders)
    "plantvillage": ("abdallahalidev/plantvillage-dataset",
                     ["Potato___Early_blight", "Potato___Late_blight", "Potato___healthy"],
                     lambda p: not any(s in p.as_posix().lower() for s in ("grayscale", "segmented"))),
    "plantdoc": ("nirmalsankalana/plantdoc-dataset",
                 ["Potato_leaf_early_blight", "Potato_leaf_late_blight"], lambda p: True),
    "pld_pakistan": ("rizwan123456789/potato-disease-leaf-datasetpld",
                     ["Early_Blight", "Late_Blight", "Healthy"], lambda p: "PLD_3_Classes" in p.as_posix()),
    "uncontrolled_indonesia": ("warcoder/potato-leaf-disease-dataset",
                               ["Bacteria", "Fungi", "Healthy", "Nematode", "Pest", "Phytopthora", "Virus"],
                               lambda p: "Uncontrolled" in p.as_posix()),
}

def locate_sources():
    """One walk over /kaggle/input: {source: {class: [folders]}}."""
    wanted = {c for _, cls, _ in POTATO_SOURCES.values() for c in cls}
    found = {s: defaultdict(list) for s in POTATO_SOURCES}
    for r, dirs, _ in os.walk(INPUT_ROOT):
        if "results" in Path(r).parts:
            dirs[:] = []; continue
        for d in dirs:
            if d in wanted:
                p = Path(r) / d
                for s, (_, cls, ok) in POTATO_SOURCES.items():
                    if d in cls and ok(p):
                        found[s][d].append(p)
    return found

def internet_ok(host="huggingface.co"):
    try:
        socket.create_connection((host, 443), timeout=5).close(); return True
    except OSError:
        return False

FOUND = locate_sources()
checks = {"GPU": (torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2 (or P100)"),
          "Internet": (internet_ok(), "Settings -> Internet -> On (pretrained weights)")}
for s, (ref, cls, _) in POTATO_SOURCES.items():
    checks[f"dataset {s}"] = (len(FOUND[s]) > 0, f"Add Input -> Datasets -> {ref}")
bad = []
for name, (ok, fix) in checks.items():
    print(f"  [{'OK' if ok else 'MISSING'}] {name}" + ("" if ok else f"  ->  {fix}"))
    if not ok:
        bad.append(name)
if bad and not os.environ.get("ONION_ALLOW_PREFLIGHT_FAIL"):
    raise RuntimeError(f"Pre-flight failed: {bad}. Fix the items above, then Run All again.")

def stage_source(s):
    out_root = DATA_LOCAL / s
    jobs = []
    for c, folders in FOUND[s].items():
        (out_root / c).mkdir(parents=True, exist_ok=True)
        for fo in folders:
            for f in sorted(fo.rglob("*")):
                if f.suffix.lower() in IMG_EXTS:
                    jobs.append((f, out_root / c / f"{f.parent.parent.name}-{f.parent.name}__{f.name}".replace(" ", "_")))
    with ThreadPoolExecutor(NUM_WORKERS) as ex:
        list(ex.map(lambda j: None if j[1].exists() else shutil.copy2(j[0], j[1]), jobs))
    return out_root

DATA_DIRS = {s: stage_source(s) for s in DATASETS if FOUND.get(s)}
rows = [dict(dataset=s, source_class=c.name, images=sum(1 for f in c.iterdir() if f.suffix.lower() in IMG_EXTS),
             folders=len(FOUND[s][c.name])) for s, root in DATA_DIRS.items() for c in sorted(root.iterdir()) if c.is_dir()]
src_tab = pd.DataFrame(rows)
src_tab.to_csv(TABLES / "source_class_counts.csv", index=False)
report("Source datasets (before any cleaning)", md_table(src_tab, "{:.0f}") +
       "\n\nSources: " + "; ".join(f"`{s}` = {POTATO_SOURCES[s][0]} ({REGION[s]})" for s in DATA_DIRS))
assert len(DATA_DIRS) == len(DATASETS) or SMOKE, f"missing datasets: {[d for d in DATASETS if d not in DATA_DIRS]}"

In [ ]:
RF_STEM = re.compile(r"^(?P<orig>.+?)_(?:jpe?g|png|bmp|webp)\.rf\.[0-9a-z]+$", re.IGNORECASE)   # (no Roboflow data here)
for d in DATASETS:
    print(f"{d:24s} -> {DATA_DIRS.get(d, 'NOT FOUND')}")

### 2.7 Index every image (dataset, class, original photo, Roboflow split)

In [ ]:
rows = []
for ds, root in DATA_DIRS.items():
    for cls_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        for f in sorted(cls_dir.rglob("*")):
            if f.suffix.lower() not in IMG_EXTS:
                continue
            r = dict(path=str(f), dataset=ds, source_class=cls_dir.name, rel=f"{ds}/{cls_dir.name}/{f.name}")
            if ds.startswith("roboflow"):
                parts = f.stem.split("__")
                r["rf_stem"] = parts[0]
                r["rf_split"] = parts[1] if len(parts) > 1 else "unknown"
                m = RF_STEM.match(parts[0])
                r["orig"] = m.group("orig") if m else parts[0]
            else:
                r["rf_stem"], r["rf_split"], r["orig"] = f.stem, "", f.stem
            rows.append(r)
idx_all = pd.DataFrame(rows)
idx_all["group"] = idx_all.dataset + "/" + idx_all.orig
print(idx_all.groupby("dataset").size().to_string())

In [ ]:
# Row order = v1's Linux order (dataset order, then case-sensitive class folder and file name). A no-op on Kaggle;
# it makes the splits identical on any OS (Windows sorts paths case-insensitively).
idx_all = (idx_all.assign(_ds=idx_all.dataset.map({d: i for i, d in enumerate(DATASETS)}),
                          _fn=idx_all.rel.str.rsplit("/", n=1).str[-1])
                  .sort_values(["_ds", "source_class", "_fn"], kind="stable")
                  .drop(columns=["_ds", "_fn"]).reset_index(drop=True))

In [ ]:
# no Roboflow exports in this benchmark: every file is its own photo (duplicates are handled by section 3)
idx = idx_all.reset_index(drop=True)
if SMOKE:
    idx = (idx.sample(frac=1.0, random_state=0).groupby(["dataset", "source_class"]).head(SMOKE_PER_CLASS)
              .sort_values("rel").reset_index(drop=True))
print("images:", len(idx))

## 3. Load images once (RAM cache) and deduplicate

1. Every image is EXIF-corrected, converted to RGB and resized to `CACHE_SIZE`². All models train from this single cache, so preprocessing is identical for every model.
2. **Exact duplicates** (same file bytes): one copy is kept; if the copies carry *different* labels, all of them are dropped.
3. **Near-duplicates** (DCT perceptual hash, Hamming ≤ `DEDUP_HAMMING`), within and across datasets, are joined into clusters (union–find):
   - cluster with conflicting harmonized labels → dropped,
   - cluster spanning datasets → kept only in the dataset holding most of the cluster (so a held-out set never shares an image with training),
   - otherwise → all members get the **same group**, so they always land on the same side of a split.

### 3.0 Image loader

In [ ]:
def load_one(path):
    raw = Path(path).read_bytes()
    with Image.open(io.BytesIO(raw)) as im:
        im = ImageOps.exif_transpose(im).convert("RGB").resize((CACHE_SIZE, CACHE_SIZE), Image.BILINEAR)
        arr = np.asarray(im, dtype=np.uint8).transpose(2, 0, 1)
    return arr, hashlib.md5(raw).hexdigest()

### 3.1 Load all images into the RAM cache (runs once, then reused)

In [ ]:
cache_file = CACHE_DIR / f"cache_{CACHE_SIZE}_{len(idx)}_{hashlib.md5(''.join(idx.rel).encode()).hexdigest()[:10]}.npy"
md5_file = cache_file.with_suffix(".md5.json")
if cache_file.exists() and md5_file.exists():
    CACHE = np.load(cache_file, mmap_mode=None)
    idx["md5"] = json.loads(md5_file.read_text())
else:
    CACHE = np.empty((len(idx), 3, CACHE_SIZE, CACHE_SIZE), dtype=np.uint8)
    md5s = [None] * len(idx)
    def _work(i):
        CACHE[i], md5s[i] = load_one(idx.path[i])
    with ThreadPoolExecutor(NUM_WORKERS) as ex:
        list(tqdm(ex.map(_work, range(len(idx))), total=len(idx), desc="loading"))
    idx["md5"] = md5s
    np.save(cache_file, CACHE); md5_file.write_text(json.dumps(md5s))
print("cache:", CACHE.shape, f"{CACHE.nbytes / 1e9:.2f} GB")

### 3b. Harmonized labels (needed by the dedup conflict rule)

In [ ]:
LABEL_MAP = {
    "plantvillage": {"Potato___Early_blight": "early_blight", "Potato___Late_blight": "late_blight", "Potato___healthy": "healthy"},
    "plantdoc": {"Potato_leaf_early_blight": "early_blight", "Potato_leaf_late_blight": "late_blight"},
    "pld_pakistan": {"Early_Blight": "early_blight", "Late_Blight": "late_blight", "Healthy": "healthy"},
    "uncontrolled_indonesia": {"Phytopthora": "late_blight", "Healthy": "healthy", "Fungi": "fungi_other",
                               "Bacteria": "bacterial", "Virus": "viral", "Pest": "pest", "Nematode": "nematode"},
}
idx["label_name"] = [LABEL_MAP.get(d, {}).get(c) for d, c in zip(idx.dataset, idx.source_class)]
unmapped = idx[idx.label_name.isna()].groupby(["dataset", "source_class"]).size()
if len(unmapped):
    print("WARNING unmapped classes (dropped):\n", unmapped)
idx["drop_reason"] = np.where(idx.label_name.isna(), "unmapped_class", "")

### 3.3 Exact duplicates (identical file bytes)

In [ ]:
# ---------- exact duplicates
dup_txt = []
g = idx[idx.drop_reason == ""].groupby("md5")
conflict_md5 = {m for m, s in g.label_name.nunique().items() if s > 1}
idx.loc[idx.md5.isin(conflict_md5), "drop_reason"] = "exact_dup_label_conflict"
dupmask = idx.drop_reason.eq("") & idx.duplicated("md5", keep="first")
idx.loc[dupmask, "drop_reason"] = "exact_dup_extra_copy"
dup_txt.append(f"- exact duplicates: {len(conflict_md5)} sets with conflicting labels (all copies dropped), "
               f"{int(dupmask.sum())} extra copies dropped")
for ds, n in idx[idx.drop_reason.str.startswith("exact")].groupby("dataset").size().items():
    dup_txt.append(f"  - {ds}: {n} images dropped")

### 3.4 Perceptual hash (pHash) of every image

In [ ]:
# ---------- perceptual hash (DCT pHash, 64 bit) from the cache
def phash_batch(arr):
    """arr: N x 3 x S x S uint8 -> N uint64 pHash (32x32 grayscale DCT, 8x8 low band vs median)."""
    t = torch.from_numpy(arr).float()
    gray = (0.299 * t[:, 0] + 0.587 * t[:, 1] + 0.114 * t[:, 2]).unsqueeze(1)
    small = F.interpolate(gray, size=(32, 32), mode="area").squeeze(1).numpy()
    d = dct(dct(small, axis=1, norm="ortho"), axis=2, norm="ortho")[:, :8, :8].reshape(len(arr), 64)
    bits = d > np.median(d, axis=1, keepdims=True)
    return np.packbits(bits, axis=1).view(">u8").ravel().astype(np.uint64)

alive = np.where(idx.drop_reason.eq(""))[0]
H = np.concatenate([phash_batch(CACHE[alive[i:i + 2048]]) for i in range(0, len(alive), 2048)])
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

### 3.5 Find near-duplicate pairs and build clusters (union–find)

In [ ]:
parent = np.arange(len(alive))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
Hb = H.view(np.uint8).reshape(-1, 8)
n_pairs = 0
for s in tqdm(range(0, len(H), 512), desc="pHash pairs"):
    x = np.bitwise_xor(Hb[s:s + 512, None, :], Hb[None, :, :])        # B x N x 8
    dist = POP[x].sum(-1, dtype=np.uint16)
    ii, jj = np.nonzero(dist <= DEDUP_HAMMING)
    for i, j in zip(ii + s, jj):
        if i < j:
            ri, rj = find(i), find(j)
            if ri != rj:
                parent[rj] = ri
            n_pairs += 1
roots = np.array([find(i) for i in range(len(alive))])
sub = idx.loc[alive, ["dataset", "label_name", "group"]].copy()
sub["cluster"] = roots
csize = sub.cluster.map(sub.cluster.value_counts())
multi = sub[csize > 1]
dup_txt.append(f"- near-duplicate pairs (Hamming <= {DEDUP_HAMMING}): {n_pairs}; clusters with >1 image: {multi.cluster.nunique()}")

### 3.6 Apply the dedup rules (label conflicts, cross-dataset copies, group merging)

In [ ]:
# conflicting labels -> drop cluster
lab_conf = multi.groupby("cluster").label_name.nunique()
bad = set(lab_conf[lab_conf > 1].index)
idx.loc[sub.index[sub.cluster.isin(bad)], "drop_reason"] = "near_dup_label_conflict"
dup_txt.append(f"- near-duplicate clusters with conflicting labels: {len(bad)} "
               f"({int(sub.cluster.isin(bad).sum())} images dropped)")

# cross-dataset -> keep only in the majority dataset
cross = multi[~multi.cluster.isin(bad)].groupby("cluster").dataset.nunique()
n_cross_drop = 0
for c in cross[cross > 1].index:
    members = sub[sub.cluster == c]
    keep_ds = members.dataset.value_counts().index[0]
    drop_ix = members.index[members.dataset != keep_ds]
    idx.loc[drop_ix, "drop_reason"] = "near_dup_cross_dataset"
    n_cross_drop += len(drop_ix)
dup_txt.append(f"- clusters spanning datasets: {int((cross > 1).sum())} ({n_cross_drop} images dropped from the minority dataset)")
pairs_by_ds = multi.groupby("cluster").dataset.agg(lambda s: "+".join(sorted(set(s))))
for k, v in pairs_by_ds[pairs_by_ds.str.contains(r"\+")].value_counts().items():
    dup_txt.append(f"  - {k}: {v} clusters")

# within dataset -> merge groups: union-find over GROUPS, so every crop of every photo in a cluster moves together
gparent = {}
def gfind(a):
    gparent.setdefault(a, a)
    while gparent[a] != a:
        gparent[a] = gparent[gparent[a]]; a = gparent[a]
    return a
still_kept = idx.loc[sub.index, "drop_reason"].eq("").values
for _, members in sub[(csize > 1).values & still_kept].groupby("cluster").group:
    gs = sorted(set(members))
    for other in gs[1:]:
        ra, rb = gfind(gs[0]), gfind(other)
        if ra != rb:
            gparent[rb] = ra
n_merged = sum(1 for g in list(gparent) if gfind(g) != g)
idx["group"] = [("merged/" + gfind(g)) if g in gparent else g for g in idx.group]
dup_txt.append(f"- photo groups merged because of near-duplicates: {n_merged}")
report("Deduplication", "\n".join(dup_txt))

## 4. Final label set and sanity grid

Classes with fewer than `MIN_CLASS_IMAGES` images are dropped. **Look at the grid**: every row should show the same disease across datasets. If *Iris yellow virus* (COLD) and *Virosis-D* (TOM2024) look like different symptoms, set `MERGE_VIRAL = False`.

In [ ]:
counts = idx[idx.drop_reason == ""].label_name.value_counts()
small = set(counts[counts < MIN_CLASS_IMAGES].index)
idx.loc[idx.label_name.isin(small) & idx.drop_reason.eq(""), "drop_reason"] = "class_too_small"
meta = idx[idx.drop_reason == ""].copy()
CLASSES = sorted(meta.label_name.unique())
C2I = {c: i for i, c in enumerate(CLASSES)}
idx["y"] = idx.label_name.map(C2I).fillna(-1).astype(int)
meta["y"] = meta.label_name.map(C2I)
DS2I = {d: i for i, d in enumerate(DATASETS)}
Y_ALL = torch.from_numpy(idx.y.values)
D_ALL = torch.from_numpy(idx.dataset.map(DS2I).values)

tab = pd.crosstab(meta.label_name, meta.dataset).reindex(columns=[d for d in DATASETS if d in DATA_DIRS])
tab["total"] = tab.sum(axis=1)
tab["n_sources"] = (tab.drop(columns="total") > 0).sum(axis=1)
report("Harmonized class counts (after dedup)", md_table(tab, "{:.0f}") +
       f"\n\nDropped images by reason: {idx.drop_reason.replace('', 'kept').value_counts().to_dict()}")
idx.to_csv(TABLES / "manifest_dedup.csv", index=False)

### 4.1 Sanity grid: does each harmonized class look the same in every dataset?

In [ ]:
# sanity grid: rows = classes present in >1 dataset, columns = up to 3 samples per dataset
shared = [c for c in CLASSES if tab.loc[c, "n_sources"] > 1]
dss = [d for d in DATASETS if d in DATA_DIRS]
fig, axes = plt.subplots(len(shared), 3 * len(dss), figsize=(2 * 3 * len(dss), 2.1 * len(shared)), squeeze=False)
rng = np.random.default_rng(0)
for r, c in enumerate(shared):
    for k, d in enumerate(dss):
        pool = meta.index[(meta.label_name == c) & (meta.dataset == d)].values
        pick = rng.choice(pool, min(3, len(pool)), replace=False) if len(pool) else []
        for j in range(3):
            ax = axes[r, 3 * k + j]; ax.axis("off")
            if j < len(pick):
                ax.imshow(CACHE[pick[j]].transpose(1, 2, 0))
            if r == 0 and j == 0:
                ax.set_title(d, fontsize=9, loc="left")
        axes[r, 0].text(-0.1, 0.5, c, transform=axes[r, 0].transAxes, ha="right", va="center", fontsize=10)
plt.tight_layout(); plt.savefig(FIGS / "label_harmonization_grid.png", dpi=110); plt.show()

## 5. Splits: LODO + stratified group 5-fold, with leakage checks

- **LODO** (RQ1/RQ2): train on 3 datasets, test on the 4th. Inner validation (early stopping, temperature scaling, conformal calibration) = one group-stratified fold of the training pool. The held-out set is split into **ID** (classes seen in training; used for accuracy/F1) and **unseen-class** images (the model cannot know them; used to test whether gating withholds them).
- **CV** (in-domain reference): stratified (dataset × class) group 5-fold over all four datasets. In-domain F1 for dataset *D* is computed on its own test folds, restricted to the same classes as LODO-*D*, so **degradation = in-domain - LODO** compares like with like.
- Assertions: no group is in two partitions, and no image id repeats.

In [ ]:
def strat_key(df):
    return (df.dataset + "|" + df.label_name).astype("category").cat.codes.values

def inner_split(pool_ix, seed=0):
    df = meta.loc[pool_ix]
    sgk = StratifiedGroupKFold(n_splits=VAL_FOLDS, shuffle=True, random_state=seed)
    tr, va = next(sgk.split(df, strat_key(df), df.group))
    return df.index.values[tr], df.index.values[va]

def check_disjoint(*parts):
    for a in range(len(parts)):
        for b in range(a + 1, len(parts)):
            ga, gb = set(idx.group[parts[a]]), set(idx.group[parts[b]])
            assert not (ga & gb), f"GROUP LEAK between partitions {a} and {b}: {list(ga & gb)[:3]}"
            assert not (set(parts[a]) & set(parts[b])), "index overlap"

### 5.1 Leave-one-dataset-out (LODO) splits

In [ ]:
SPLITS = {}
for held in dss:
    pool = meta.index[meta.dataset != held].values
    test = meta.index[meta.dataset == held].values
    tr, va = inner_split(pool)
    train_classes = set(meta.y[tr])
    va = va[np.isin(meta.y[va], list(train_classes))]
    te_id = test[np.isin(meta.y[test], list(train_classes))]
    te_ood = test[~np.isin(meta.y[test], list(train_classes))]
    check_disjoint(tr, va, test)
    SPLITS[("lodo", held)] = dict(train=tr, val=va, test=test, test_id=te_id, test_ood=te_ood)

### 5.1b Inner leave-one-*source*-out splits (for cross-source conformal calibration)

For held-out set *H*, the training sources are the other three. For each training source *S*, an **inner model** is trained on the remaining two sources and predicts *S*, which it has never seen. This measures how much standard conformal coverage drops on **a region the model was not trained on**, which is exactly the deployment situation. From it we learn how much stricter the conformal level must be (α′ < α) to keep 90% coverage. The main model then uses α′ on its own validation scores (section 10). *H* is never touched.

In [ ]:
for held in dss:
    sources = [d for d in dss if d != held]
    for src in sources:
        pool = meta.index[meta.dataset.isin([d for d in sources if d != src])].values
        test = meta.index[meta.dataset == src].values
        tr, va = inner_split(pool)
        train_classes = set(meta.y[tr])
        va = va[np.isin(meta.y[va], list(train_classes))]
        te_id = test[np.isin(meta.y[test], list(train_classes))]
        assert not set(meta.loc[np.concatenate([tr, va, test])].dataset) & {held}, "held-out set leaked into inner split"
        check_disjoint(tr, va, test)
        SPLITS[("inner", f"{held}|{src}")] = dict(train=tr, val=va, test=test, test_id=te_id,
                                                   test_ood=test[~np.isin(meta.y[test], list(train_classes))])

### 5.2 Stratified group 5-fold CV (in-domain reference)

In [ ]:
sgk = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
for k, (pool_i, te_i) in enumerate(sgk.split(meta, strat_key(meta), meta.group)):
    pool, test = meta.index.values[pool_i], meta.index.values[te_i]
    tr, va = inner_split(pool, seed=k)
    va = va[np.isin(meta.y[va], list(set(meta.y[tr])))]
    check_disjoint(tr, va, test)
    SPLITS[("cv", f"fold{k}")] = dict(train=tr, val=va, test=test, test_id=test, test_ood=test[:0])

### 5.3b Reuse v1's exact splits when v1's output is attached
`StratifiedGroupKFold` folds can change between scikit-learn / NumPy versions. When the attached v1 output contains `split_membership.csv` for exactly these images (same row ids and files), the splits are **read from it**. v2 is then evaluated on the identical partitions and the v1 runs stay comparable. Otherwise the splits computed above are used.

In [ ]:
SPLITS_FROM_V1 = False
mem_files = sorted(INPUT_ROOT.rglob("results/tables/split_membership.csv")) if INPUT_ROOT.exists() and not SMOKE else []
for mf in mem_files:
    m1 = pd.read_csv(mf, index_col=0, keep_default_na=False)
    if len(m1) != len(meta) or not (m1.index.values == meta.index.values).all() or not (m1.rel.values == meta.rel.values).all():
        print("v1 split file does not match these images:", mf); continue
    new = {}
    for (proto, name), s in SPLITS.items():
        col = f"{proto}_{name}"
        if col not in m1:
            break
        v = m1[col].astype(str).values
        tr, va, te = (m1.index.values[v == p] for p in ("train", "val", "test"))
        seen = list(set(meta.y[tr]))
        check_disjoint(tr, va, te)
        new[(proto, name)] = dict(train=tr, val=va, test=te,
                                  test_id=te if proto == "cv" else te[np.isin(meta.y[te], seen)],
                                  test_ood=te[:0] if proto == "cv" else te[~np.isin(meta.y[te], seen)])
    else:
        SPLITS, SPLITS_FROM_V1 = new, True
        print("splits loaded from v1:", mf); break
print("splits from v1:", SPLITS_FROM_V1)

### 5.3 Split summary

In [ ]:
rows = []
for (proto, name), s in SPLITS.items():
    rows.append(dict(protocol=proto, split=name, train=len(s["train"]), val=len(s["val"]),
                     test=len(s["test"]), test_id=len(s["test_id"]), test_unseen_class=len(s["test_ood"]),
                     train_classes=len(set(meta.y[s["train"]])),
                     shared_test_classes=", ".join(CLASSES[c] for c in sorted(set(meta.y[s["test_id"]])))))
split_df = pd.DataFrame(rows)
split_df.to_csv(TABLES / "splits_summary.csv", index=False)
report("Splits (all leakage checks passed)", md_table(split_df, "{:.0f}"))

### 5.4 Save split signature and per-image split membership (supplementary material)

In [ ]:
split_sig = hashlib.md5(json.dumps({f"{k[0]}/{k[1]}": sorted(map(int, v["test"])) for k, v in SPLITS.items()}).encode()).hexdigest()
# a fine-tuned run is reused only if it was trained on exactly these splits with the same budget (never smoke runs)
RUN_TAG = hashlib.md5(f"{split_sig}|{EPOCHS}|{IMG_SIZE}|{SMOKE}".encode()).hexdigest()[:12]
# v1 runs carry no tag: they are accepted only if the attached v1 output has the SAME split signature
v1_sigs = {p.read_text().strip() for p in INPUT_ROOT.rglob("results/split_signature.txt")} if INPUT_ROOT.exists() else set()
LEGACY_OK = (split_sig in v1_sigs) and not SMOKE
print(f"split signature {split_sig} | attached v1 signatures {v1_sigs or 'none'} | v1 runs reusable: {LEGACY_OK}")
if v1_sigs and not LEGACY_OK and not SMOKE:
    print("WARNING: the attached v1 output was made with different splits -> its runs are ignored (fine-tuned baselines will be retrained).")
(RES / "split_signature.txt").write_text(split_sig)
mem = pd.DataFrame(index=meta.index)
for (proto, name), s in SPLITS.items():
    col = pd.Series("", index=meta.index)
    for part in ("train", "val", "test"):
        col[s[part]] = part
    mem[f"{proto}_{name}"] = col
pd.concat([meta[["rel", "dataset", "label_name", "group"]], mem], axis=1).to_csv(TABLES / "split_membership.csv")

## 6. Shared helpers: remedy groups, calibration, run files
A farmer acts on a **treatment**, not on a disease name. Diseases that share one management action form one remedy group, so confusing them is harmless; confusing groups (e.g. viral → thrips control vs fungal → fungicide) is a **wrong remedy**.

In [ ]:
import copy, gc
from collections import deque

# REMEDY_GROUP (disease -> management action group) and ACTION (group -> advice template) come from the config cell
GROUPS = sorted(set(REMEDY_GROUP.values()))
G2I = {g: i for i, g in enumerate(GROUPS)}
CLASS_GROUP = np.array([G2I[REMEDY_GROUP[c]] for c in CLASSES])     # global class id -> remedy group id

def batches(ix, bs):
    for i in range(0, len(ix), bs):
        yield ix[i:i + bs]

def softmax(z, T=1.0):
    z = z / T; z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def fit_temperature(logits, y):
    """Temperature scaling on the source validation split (bounded; rejected if it does not lower val NLL)."""
    lg = torch.tensor(logits, dtype=torch.float64); yt = torch.tensor(y)
    nll = lambda T: float(F.cross_entropy(lg / T, yt))
    t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([t], lr=0.05, max_iter=100, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(lg / t.clamp(-3, 3).exp(), yt); loss.backward(); return loss
    try:
        opt.step(closure); T = float(t.detach().clamp(-3, 3).exp())
    except Exception:
        T = 1.0
    if not np.isfinite(T) or nll(T) > nll(1.0):
        T = 1.0
    return T

def ece(p, y, bins=15):
    conf, pred = p.max(1), p.argmax(1); e = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e

def conf_quantile(scores, alpha):
    n = len(scores)
    if n == 0:
        return 1.0
    return float(np.quantile(scores, min(1.0, math.ceil((n + 1) * (1 - alpha)) / n), method="higher"))

def aurc(conf, correct):
    o = np.argsort(-conf); err = 1 - correct[o].astype(float)
    return float(np.mean(np.cumsum(err) / np.arange(1, len(err) + 1))) if len(err) else np.nan

def group_probs(p, classes):
    """Disease probabilities (over the run's classes) -> remedy-group probabilities."""
    M = np.zeros((len(classes), len(GROUPS)))
    M[np.arange(len(classes)), CLASS_GROUP[classes]] = 1
    return p @ M

def cdf_of(ref, x):
    """Empirical CDF of `ref` evaluated at x: turns any risk score into 'validation-quantile units'."""
    r = np.sort(np.asarray(ref))
    return np.searchsorted(r, x, side="right") / max(1, len(r))

def f1_per_class_np(y, p, labels):
    labels = np.asarray(labels)
    top = int(max(y.max(initial=0), p.max(initial=0), labels.max(initial=0))) + 1
    pos = np.full(top + 1, len(labels)); pos[labels] = np.arange(len(labels))
    n = len(labels) + 1
    cm = np.bincount(pos[y] * n + pos[p], minlength=n * n).reshape(n, n)
    tp = np.diag(cm)[:-1]; fp = cm[:, :-1].sum(0) - tp; fn = cm[:-1, :].sum(1) - tp
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)

def f1_macro_np(y, p, labels):
    """Fast macro-F1 (same definition as sklearn f1_score(labels=labels, average='macro'))."""
    return float(f1_per_class_np(np.asarray(y), np.asarray(p), labels).mean()) if len(labels) else np.nan

def holm(p):
    p = np.asarray(p, dtype=float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(o):
        run = max(run, min(1.0, (len(p) - r) * p[i])); adj[i] = run
    return adj

def run_key(model_name, proto, split, seed):
    return re.sub(r"[^A-Za-z0-9_.+-]", "_", f"{model_name}__{proto}__{split}__s{seed}")

def run_done(key):
    """v1-format run (fine-tuned baselines): exists AND was made with the current splits/budget. Runs from v2 carry
    RUN_TAG; untagged v1 runs count only if the attached v1 split signature equals ours (LEGACY_OK) and epochs match."""
    f = RUNS / f"{key}.json"
    if not f.exists() or not (RUNS / f"{key}.npz").exists():
        return False
    try:
        j = json.loads(f.read_text())
    except Exception:
        return False
    if "run_tag" in j:
        return j["run_tag"] == RUN_TAG
    return LEGACY_OK and j.get("epochs") == EPOCHS

def uid_of(i):
    return hashlib.md5(idx.rel[i].encode()).hexdigest()[:12]

## 7. Multi-scale tile features from frozen foundation encoders (one pass) + acquisition metadata
Every image is resized to a 672×672 canvas (the same squash as v1's cache) and cut into **14 views**: the global view (= what v1 models saw), a 2×2 grid and a 3×3 grid, each resized to 224². All encoders read the same views in **one decoding pass**. Features (float16) are cached in `results/features` and reused on resume. The same pass records **acquisition metadata** (native resolution, file size, JPEG quantization step, sharpness, colour statistics) for the shortcut diagnosis.

Why tiles: v1 showed that sources differ in *scale* (whole-plant photos with soil vs tight, often pixelated box crops). At 224² a lesion in a whole-plant photo covers a few pixels; the 3×3 tiles show it at roughly box-crop scale.

In [ ]:
def tile_boxes(S=TILE_CANVAS, grids=TILE_GRIDS):
    boxes, scale = [], []
    for g in grids:
        e = np.linspace(0, S, g + 1).round().astype(int)
        for r in range(g):
            for c in range(g):
                boxes.append((int(e[c]), int(e[r]), int(e[c + 1]), int(e[r + 1]))); scale.append(g)
    return boxes, np.array(scale)

TILE_BOXES, TILE_SCALE = tile_boxes()
N_TILES = len(TILE_BOXES)
SCALE_IDX = torch.tensor([list(TILE_GRIDS).index(g) for g in TILE_SCALE])

def load_tiles(path):
    """Original file -> (N_TILES x 224 x 224 x 3 uint8 views, acquisition metadata)."""
    raw = Path(path).read_bytes()
    with Image.open(io.BytesIO(raw)) as im0:
        fmt, (W0, H0) = im0.format, im0.size
        q = float(np.mean(im0.quantization[0])) if fmt == "JPEG" and getattr(im0, "quantization", None) else np.nan
        if fmt == "JPEG":
            im0.draft("RGB", (2 * TILE_CANVAS, 2 * TILE_CANVAS))      # fast decode of large JPEGs
        im = ImageOps.exif_transpose(im0).convert("RGB")
    canvas = im.resize((TILE_CANVAS, TILE_CANVAS), Image.BILINEAR)
    tiles = np.stack([np.asarray(canvas.crop(b).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) for b in TILE_BOXES])
    a = np.asarray(canvas.resize((256, 256), Image.BILINEAR), dtype=np.float32) / 255.0
    gray = a @ np.array([0.299, 0.587, 0.114], np.float32)
    lap = 4 * gray[1:-1, 1:-1] - gray[:-2, 1:-1] - gray[2:, 1:-1] - gray[1:-1, :-2] - gray[1:-1, 2:]
    mx, mn = a.max(-1), a.min(-1)
    meta_ = dict(native_long=max(W0, H0), native_short=min(W0, H0), file_kb=len(raw) / 1024,
                 is_jpeg=float(fmt == "JPEG"), jpeg_qstep=q, sharpness=float(np.log(lap.var() + 1e-8)),
                 brightness=float(gray.mean()), saturation=float(((mx - mn) / (mx + 1e-6)).mean()),
                 green_frac=float(((a[..., 1] > a[..., 0]) & (a[..., 1] > a[..., 2])).mean()))
    return tiles, meta_

def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

def load_encoder(spec):
    if spec["kind"] == "timm":
        m = timm.create_model(spec["name"], pretrained=PRETRAINED, num_classes=0, img_size=IMG_SIZE)
        cfg = timm.data.resolve_data_config({}, model=m)
        return m, cfg["mean"], cfg["std"], m
    import open_clip
    if not PRETRAINED:
        raise RuntimeError("open_clip model needs pretrained weights (Internet)")
    model, _, _ = open_clip.create_model_and_transforms(spec["name"])
    enc = model.visual
    class _Wrap(nn.Module):
        def __init__(s, v): super().__init__(); s.v = v
        def forward(s, x): return s.v(x)
    return _Wrap(enc), open_clip.OPENAI_DATASET_MEAN, open_clip.OPENAI_DATASET_STD, enc

In [ ]:
FM_PARAMS = {}

@torch.no_grad()
def extract_tiles(paths, n_rows, rows, tag):
    """Run every encoder in FM_ENCODERS over the 14 views of each image (one decoding pass).
    Writes FEATS/tiles_<encoder>_<tag>.npy (n_rows x N_TILES x D, float16; row = image row id) and
    FEATS/acqmeta_<tag>.csv. Already-finished encoders are skipped (resume)."""
    rows = np.asarray(rows)
    files = {fm: FEATS / f"tiles_{fm}_{tag}.npy" for fm in FM_ENCODERS}
    meta_f = FEATS / f"acqmeta_{tag}.csv"
    todo = [fm for fm in FM_ENCODERS if not files[fm].exists()]
    if todo or not meta_f.exists():
        encs = {}
        for fm in todo:
            try:
                enc, mean, std, counted = load_encoder(FM_ENCODERS[fm])
                encs[fm] = (enc.to(DEVICE).eval(), torch.tensor(mean, device=DEVICE).view(1, 3, 1, 1),
                            torch.tensor(std, device=DEVICE).view(1, 3, 1, 1), count_params(counted))
            except Exception as e:
                print(f"{fm}: encoder unavailable ({e}) -> skipped")
        outs, tmp, metas = {}, {}, [None] * len(paths)
        chunks = [list(range(i, min(i + 16, len(paths)))) for i in range(0, len(paths), 16)]
        with ThreadPoolExecutor(NUM_WORKERS) as ex:
            submit = lambda c: (c, [ex.submit(load_tiles, paths[j]) for j in c])
            pending = deque(submit(c) for c in chunks[:6]); nxt = 6
            for _ in tqdm(range(len(chunks)), desc=f"tiles {tag}"):
                c, futs = pending.popleft()
                if nxt < len(chunks):
                    pending.append(submit(chunks[nxt])); nxt += 1
                res = [f.result() for f in futs]
                for j, (_, m_) in zip(c, res):
                    metas[j] = m_
                if not encs:
                    continue
                x = torch.from_numpy(np.stack([t for t, _ in res])).to(DEVICE)
                x = x.flatten(0, 1).permute(0, 3, 1, 2).float() / 255.0          # (B*T) x 3 x 224 x 224
                for fm, (enc, mean, std, _) in encs.items():
                    with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                        z = torch.cat([enc((xb - mean) / std).float() for xb in x.split(112)])
                    if not torch.isfinite(z).all():                            # fp16 overflow guard
                        z = torch.cat([enc((xb - mean) / std).float() for xb in x.split(112)])
                    z = z.view(len(c), N_TILES, -1).cpu().numpy().astype(np.float16)
                    if fm not in outs:
                        tmp[fm] = CACHE_DIR / f"tiles_{fm}_{tag}.npy"
                        outs[fm] = np.lib.format.open_memmap(tmp[fm], mode="w+", dtype=np.float16,
                                                             shape=(n_rows, N_TILES, z.shape[-1]))
                    outs[fm][rows[c]] = z
        names = list(outs)
        for fm in names:
            outs[fm].flush()
        outs.clear(); gc.collect()
        for fm in names:
            shutil.move(str(tmp[fm]), str(files[fm]))
            (FEATS / f"tiles_{fm}_{tag}.params.json").write_text(json.dumps({"params_m": encs[fm][3]}))
        del encs; gc.collect(); torch.cuda.empty_cache()
        pd.DataFrame(metas, index=pd.Index(rows, name="row")).to_csv(meta_f)
    feats = {}
    for fm in FM_ENCODERS:
        if files[fm].exists():
            feats[fm] = np.load(files[fm], mmap_mode="r")
            pf = FEATS / f"tiles_{fm}_{tag}.params.json"
            if pf.exists():
                FM_PARAMS[fm] = json.loads(pf.read_text())["params_m"]
    return feats, pd.read_csv(meta_f, index_col="row")

### 7.1 Extract (or reuse) features for the benchmark

In [ ]:
ROWS = meta.index.values
TILE_TAG = hashlib.md5(("".join(idx.rel) + f"|{TILE_GRIDS}|{TILE_CANVAS}|{IMG_SIZE}").encode()).hexdigest()[:10]
TILE_FEATS, ACQ = extract_tiles(list(idx.path.values[ROWS]), len(idx), ROWS, TILE_TAG)
FMS = [fm for fm in FM_ENCODERS if fm in TILE_FEATS]
assert FMS, "no foundation encoder could be loaded (Internet on? timm >= 1.0.20?)"
if MAIN_FM not in FMS:
    print(f"WARNING: {MAIN_FM} unavailable -> using {FMS[0]} as the main encoder"); MAIN_FM = FMS[0]
if EDGE_FM not in FMS:
    EDGE_FM = MAIN_FM
print("encoders:", {fm: TILE_FEATS[fm].shape for fm in FMS}, "| params (M):", FM_PARAMS, f"| {hours():.2f} h")

### 7.2 Optional external prospective test set (`onion_external/<class>/…`)
Folders must be named with harmonized class names (`healthy`, `purple_blotch`, `stemphylium`, `downy_mildew`, `viral`, `botrytis`, `rust`, `xanthomonas`, `fusarium`, `bulb_blight`, `burn_tip`). External images are **never** used for training, calibration or model selection. Collect them yourself (new farms, new phones, both whole-plant and close-up shots of the same plant, GPS + date), not from the public sets.

In [ ]:
def find_external():
    if not INPUT_ROOT.exists():
        return None
    for p in INPUT_ROOT.rglob(EXTERNAL_NAME):
        if p.is_dir() and any(c.is_dir() for c in p.iterdir()):
            return p
    return None

EXT, EXT_FEATS = None, {}
EXT_DIR = find_external()
if EXT_DIR is not None:
    rows_ = []
    for cls_dir in sorted(d for d in EXT_DIR.iterdir() if d.is_dir()):
        lab = cls_dir.name.strip().lower().replace(" ", "_").replace("-", "_")
        for f in sorted(cls_dir.rglob("*")):
            if f.suffix.lower() in IMG_EXTS:
                rows_.append(dict(path=str(f), label_name=lab, rel=f"ext/{cls_dir.name}/{f.name}"))
    EXT = pd.DataFrame(rows_)
    bad_lab = sorted(set(EXT.label_name) - set(REMEDY_GROUP)) if len(EXT) else []
    if bad_lab:
        print("WARNING external classes not in the remedy table (dropped):", bad_lab)
    EXT = EXT[EXT.label_name.isin(REMEDY_GROUP)].reset_index(drop=True) if len(EXT) else EXT
    if len(EXT):
        EXT["y"] = EXT.label_name.map(C2I).fillna(-1).astype(int)
        EXT["g"] = EXT.label_name.map(lambda c: G2I[REMEDY_GROUP[c]])
        ext_tag = "ext_" + hashlib.md5(("".join(EXT.rel) + TILE_TAG).encode()).hexdigest()[:10]
        EXT_FEATS, EXT_ACQ = extract_tiles(list(EXT.path), len(EXT), np.arange(len(EXT)), ext_tag)
        tr_all, va_all = inner_split(meta.index.values)
        seen = list(set(meta.y[tr_all]))
        SPLITS[("ext", "all")] = dict(train=tr_all, val=va_all, test=np.arange(len(EXT)),
                                      test_id=np.where(np.isin(EXT.y, seen))[0],
                                      test_ood=np.where(~np.isin(EXT.y, seen))[0])
        report("External test set", md_table(EXT.label_name.value_counts().rename("images").to_frame(), "{:.0f}"))
    else:
        EXT = None
print("external test set:", "none" if EXT is None else f"{len(EXT)} images from {EXT_DIR}")

# optional single-source reference splits (e.g. the classic PlantVillage -> PlantDoc), from EXTRA_SPLITS in the config
for name_, tr_src, te_src in globals().get("EXTRA_SPLITS", []):
    pool = meta.index[meta.dataset.isin(tr_src)].values
    test = meta.index[meta.dataset.isin(te_src)].values
    if len(pool) == 0 or len(test) == 0:
        continue
    tr_, va_ = inner_split(pool)
    seen = list(set(meta.y[tr_]))
    va_ = va_[np.isin(meta.y[va_], seen)]
    check_disjoint(tr_, va_, test)
    SPLITS[("single", name_)] = dict(train=tr_, val=va_, test=test, test_id=test[np.isin(meta.y[test], seen)],
                                     test_ood=test[~np.isin(meta.y[test], seen)])
    print(f"single-source split {name_}: train {len(tr_)} | test {len(test)} ({len(SPLITS[('single', name_)]['test_id'])} seen classes)")

def y_of(proto, ix):
    """Global class ids of test rows (-1 = external class not in the benchmark)."""
    return EXT.y.values[np.asarray(ix)] if proto == "ext" else Y_ALL[np.asarray(ix)].numpy()

def g_of(proto, ix):
    """True remedy group of test rows."""
    return EXT.g.values[np.asarray(ix)] if proto == "ext" else CLASS_GROUP[Y_ALL[np.asarray(ix)].numpy()]

## 8. Diagnosis (RQ1): shortcut, label disagreement, or both?
- **D1, acquisition metadata only** (no pixels of the leaf): can a gradient-boosted tree tell the *source* from native resolution, file size, JPEG quantization, sharpness and colour statistics? And *within* a source, can it tell the *disease*? (If yes, the source has a class-correlated acquisition shortcut.)
- **D2, foundation features**: source identifiability of the DINOv3 global embedding, overall and **within a class** (healthy / purple blotch occur in ≥ 3 sources). Measured raw, after **class-conditional LEACE** and after plain (global) LEACE, with a linear and a non-linear (MLP) probe. A sanity check makes sure disease information survives erasure.
- **D3, label-consistency index (LCI)**: for a class *c* shared by sources A and B, LCI = cos(centroid of *c* in A, centroid of *c* in B) − max over other classes *c′* of cos(*c* in A, *c′* in B). LCI > 0 means "*c* in A looks more like *c* in B than like anything else in B". We compute it raw and after **per-source centring** (which removes a global style offset). If LCI stays ≤ 0 after centring, the two sources' labels with the same name do not describe the same visual concept. That is an *annotation/taxonomy* problem that no model can fix. Then the blinded expert audit (section 15) decides.

All probes: stratified **group** CV (photo groups never split), class-balanced loss, **balanced accuracy** (chance = 1/number of targets).

In [ ]:
class Eraser:
    """LEACE (Belrose et al., NeurIPS 2023): closed-form erasure of all LINEARLY available information about a
    categorical concept z (here: the acquisition source) from features x, with minimal change to x:
        r(x) = x - W^+ P W (x - mu),  W = Sigma_xx^{-1/2},  P = projection onto colspace(W Sigma_xz).
    y given -> CLASS-CONDITIONAL: Sigma_xx and Sigma_xz are pooled within classes (x and z centred per class), so
    only source information that is NOT explained by the class mix is removed (the class mix differs by source)."""
    def __init__(self, X, z, y=None, shrink=1e-3):
        dev = X.device
        X = X.double(); z = z.long().to(dev)
        Z = F.one_hot(z, int(z.max()) + 1).double()
        Xc, Zc = X - X.mean(0), Z - Z.mean(0)
        if y is not None:
            y = y.to(dev)
            for c in torch.unique(y):
                m = y == c
                Xc[m] = X[m] - X[m].mean(0); Zc[m] = Z[m] - Z[m].mean(0)
        n, D = X.shape
        Sxx = Xc.T @ Xc / n
        Sxx += shrink * Sxx.diagonal().mean() * torch.eye(D, dtype=Sxx.dtype, device=dev)
        Sxz = Xc.T @ Zc / n
        del Xc, Zc
        ev, V = torch.linalg.eigh(Sxx); ev = ev.clamp(min=float(ev.max()) * 1e-9)
        W = (V * ev.rsqrt()) @ V.T; Winv = (V * ev.sqrt()) @ V.T
        U, s, _ = torch.linalg.svd(W @ Sxz, full_matrices=False)
        U = U[:, s > s.max() * 1e-6] if s.numel() and float(s.max()) > 0 else U[:, :0]
        self.rank = int(U.shape[1])
        self.M = (Winv @ U @ U.T @ W).float()
        self.mu = X.mean(0).float()

    def __call__(self, X):
        sh = X.shape; X2 = X.reshape(-1, sh[-1]).float()
        return (X2 - (X2 - self.mu) @ self.M.T).reshape(sh)

def torch_probe(Xtr, ytr, Xte, hidden=0, steps=300, wd=1e-3, seed=0):
    """Class-balanced multinomial logistic regression (hidden=0) or 1-hidden-layer MLP; full-batch AdamW."""
    ytr = np.asarray(ytr)
    cls, yi = np.unique(ytr, return_inverse=True)
    if len(cls) < 2:
        return np.full(len(Xte), cls[0] if len(cls) else None, dtype=object)
    torch.manual_seed(seed)
    Xtr = torch.as_tensor(np.asarray(Xtr), dtype=torch.float32, device=DEVICE)
    Xte = torch.as_tensor(np.asarray(Xte), dtype=torch.float32, device=DEVICE)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd
    yt = torch.tensor(yi, device=DEVICE)
    w = torch.tensor(len(yi) / (len(cls) * np.bincount(yi)), dtype=torch.float32, device=DEVICE)
    D = Xtr.shape[1]
    net = (nn.Linear(D, len(cls)) if hidden == 0 else
           nn.Sequential(nn.Linear(D, hidden), nn.GELU(), nn.Dropout(0.2), nn.Linear(hidden, len(cls)))).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-2 if hidden == 0 else 3e-3, weight_decay=wd)
    net.train()
    for _ in range(steps):
        loss = F.cross_entropy(net(Xtr), yt, weight=w); opt.zero_grad(); loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        return cls[net(Xte).argmax(1).cpu().numpy()]

def bacc(y, p):
    from sklearn.metrics import balanced_accuracy_score
    m = np.array([v is not None for v in p])
    return balanced_accuracy_score(np.asarray(y)[m].astype(str), np.asarray(p)[m].astype(str)) if m.any() else np.nan

D_SRC = idx.dataset.values[ROWS]; D_LAB = idx.label_name.values[ROWS]; D_GRP = idx.group.values[ROWS]
D_STRAT = pd.Series(D_SRC + "|" + D_LAB).astype("category").cat.codes.values
D_FOLDS = list(StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=0).split(ROWS, D_STRAT, D_GRP))
SHARED3 = [c for c in CLASSES if meta[meta.label_name == c].dataset.nunique() >= min(3, len(dss))]
print("classes in >= 3 sources:", SHARED3)

### 8.1 D1: acquisition metadata (no leaf pixels)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

AM = ACQ.loc[ROWS].copy()
AM["log_long"], AM["log_short"] = np.log(AM.native_long), np.log(AM.native_short)
AM["aspect"] = AM.native_long / AM.native_short
AM["log_kb"] = np.log(AM.file_kb)
AM["jpeg_qstep"] = AM.jpeg_qstep.fillna(-1)
MCOLS = ["log_long", "log_short", "aspect", "log_kb", "is_jpeg", "jpeg_qstep", "sharpness", "brightness", "saturation", "green_frac"]
XM = AM[MCOLS].values.astype(np.float32)

acq_tab = AM.assign(dataset=D_SRC).groupby("dataset")[["native_long", "native_short", "file_kb", "jpeg_qstep",
                                                       "sharpness", "saturation", "green_frac"]].median()
acq_tab.to_csv(TABLES / "d1_acquisition_by_source.csv")
report("D1 - acquisition metadata by source (median)", md_table(acq_tab, "{:.2f}"))

def meta_probe(mask, target):
    pred = np.full(len(ROWS), None, dtype=object)
    for tr, te in D_FOLDS:
        tr, te = tr[mask[tr]], te[mask[te]]
        if len(te) == 0 or len(set(target[tr])) < 2:
            continue
        clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, class_weight="balanced", random_state=0)
        clf.fit(XM[tr], target[tr]); pred[te] = clf.predict(XM[te])
    return bacc(target[mask], pred[mask]), len(set(target[mask]))

d1 = []
for name, mask in [("all images", np.ones(len(ROWS), bool))] + [(f"within {c}", D_LAB == c) for c in SHARED3]:
    b, k = meta_probe(mask, D_SRC)
    d1.append(dict(target="source", subset=name, balanced_acc=b, chance=1 / k, n=int(mask.sum())))
for s in dss:
    mask = D_SRC == s
    if len(set(D_LAB[mask])) >= 2:
        b, k = meta_probe(mask, D_LAB)
        d1.append(dict(target="disease", subset=f"within source {s}", balanced_acc=b, chance=1 / k, n=int(mask.sum())))
d1 = pd.DataFrame(d1); d1["above_chance"] = d1.balanced_acc - d1.chance
d1.to_csv(TABLES / "d1_metadata_probes.csv", index=False)
report("D1 - what metadata alone can predict (balanced accuracy, grouped CV)", md_table(d1, "{:.3f}"))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, col, lab_ in zip(axes, ["native_short", "file_kb", "sharpness"],
                         ["native short side (px, log scale)", "file size (kB, log scale)", "sharpness (log Laplacian var.)"]):
    data = [AM[col].values[D_SRC == s] for s in dss]
    ax.boxplot(data, showfliers=False); ax.set_xticks(range(1, len(dss) + 1)); ax.set_xticklabels(dss, rotation=20, fontsize=8)
    if col != "sharpness":
        ax.set_yscale("log")
    ax.set_title(lab_, fontsize=9)
plt.tight_layout(); plt.savefig(FIGS / "d1_acquisition_by_source.png", dpi=120); plt.show()

### 8.2 D2: source identifiability of foundation features (within class), before and after erasure

In [ ]:
XG = np.asarray(TILE_FEATS[MAIN_FM][ROWS, 0], dtype=np.float32)       # global view, main encoder
src_code = pd.Series(D_SRC).map(DS2I).values
lab_code = pd.Series(D_LAB).map(C2I).values
tasks = ([("source", "all images", np.ones(len(ROWS), bool), D_SRC)] +
         [("source", f"within {c}", D_LAB == c, D_SRC) for c in SHARED3] +
         [("disease", "all images", np.ones(len(ROWS), bool), D_LAB)])
preds = {}
for fi, (tr, te) in enumerate(tqdm(D_FOLDS, desc="D2 probes")):
    mu, sd = XG[tr].mean(0), XG[tr].std(0) + 1e-6
    Xtr_t = torch.tensor((XG[tr] - mu) / sd, device=DEVICE); Xte_t = torch.tensor((XG[te] - mu) / sd, device=DEVICE)
    variants = {"raw": (Xtr_t, Xte_t)}
    for ename, yy in (("class-cond. LEACE", torch.tensor(lab_code[tr])), ("global LEACE", None)):
        er = Eraser(Xtr_t, torch.tensor(src_code[tr]), yy)
        variants[ename] = (er(Xtr_t), er(Xte_t))
    for vname, (A, B) in variants.items():
        A, B = A.cpu().numpy(), B.cpu().numpy()
        for target, subset, mask, tgt in tasks:
            mtr, mte = mask[tr], mask[te]
            for probe, hid in (("linear", 0), ("MLP", 256)):
                if target == "disease" and probe == "MLP":
                    continue
                key = (target, subset, vname, probe)
                preds.setdefault(key, np.full(len(ROWS), None, dtype=object))
                if mte.any():
                    preds[key][te[mte]] = torch_probe(A[mtr], tgt[tr][mtr], B[mte], hidden=hid)
d2 = []
for (target, subset, vname, probe), p in preds.items():
    mask = next(m for t, s, m, _ in tasks if t == target and s == subset)
    tgt = D_SRC if target == "source" else D_LAB
    d2.append(dict(target=target, subset=subset, features=vname, probe=probe,
                   balanced_acc=bacc(tgt[mask], p[mask]), chance=1 / len(set(tgt[mask]))))
d2 = pd.DataFrame(d2)
d2.to_csv(TABLES / "d2_feature_source_probes.csv", index=False)
d2p = d2.pivot_table(index=["target", "subset", "probe"], columns="features", values="balanced_acc")
d2p = d2p.join(d2.groupby(["target", "subset", "probe"]).chance.first())
report(f"D2 - source / disease identifiability from {MAIN_FM} global features (balanced accuracy, grouped CV)", md_table(d2p))

### 8.3 D3: label-consistency index across sources

In [ ]:
def l2n(a):
    return a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)

ZG = l2n((XG - XG.mean(0)) / (XG.std(0) + 1e-6))
ZC = ZG.copy()
for s in dss:                                     # per-source centring removes a global style offset
    m = D_SRC == s
    ZC[m] = ZG[m] - ZG[m].mean(0)
ZC = l2n(ZC)

def centroids(Z, min_n=10):
    out = {}
    for (s, c), ix in pd.Series(np.arange(len(ROWS))).groupby([D_SRC, D_LAB]).groups.items():
        if len(ix) >= min_n:
            out[(s, c)] = l2n(Z[np.asarray(ix)].mean(0, keepdims=True))[0]
    return out

lci_rows, transfer_rows = [], []
for space, Z in (("raw", ZG), ("source-centred", ZC)):
    cen = centroids(Z)
    for c in CLASSES:
        srcs = [s for s in dss if (s, c) in cen]
        for A in srcs:
            for B in srcs:
                if A == B:
                    continue
                others = [c2 for (s2, c2) in cen if s2 == B and c2 != c]
                if not others:
                    continue
                same = float(cen[(A, c)] @ cen[(B, c)])
                best_other = max(others, key=lambda c2: float(cen[(A, c)] @ cen[(B, c2)]))
                lci_rows.append(dict(space=space, cls=c, src_A=A, src_B=B, cos_same=same,
                                     cos_best_other=float(cen[(A, c)] @ cen[(B, best_other)]),
                                     nearest_other_class=best_other, LCI=same - float(cen[(A, c)] @ cen[(B, best_other)])))
    # nearest-centroid transfer: classify A's images with B's class centroids (shared classes only)
    for A in dss:
        for B in dss:
            if A == B:
                continue
            shared = [c for c in CLASSES if (A, c) in cen and (B, c) in cen]
            if len(shared) < 2:
                continue
            m = (D_SRC == A) & np.isin(D_LAB, shared)
            C = np.stack([cen[(B, c)] for c in shared])
            pred = np.array(shared)[(Z[m] @ C.T).argmax(1)]
            transfer_rows.append(dict(space=space, train_src=B, test_src=A, shared=", ".join(shared),
                                      macro_f1=f1_score(D_LAB[m], pred, labels=shared, average="macro"), chance=1 / len(shared)))
lci = pd.DataFrame(lci_rows); lci.to_csv(TABLES / "d3_label_consistency_pairs.csv", index=False)
if len(lci):
    lsum = lci.pivot_table(index="cls", columns="space", values="LCI", aggfunc="mean")
    lsum["pairs"] = lci[lci.space == "raw"].groupby("cls").size()
    def verdict(r):
        if r.get("source-centred", np.nan) > 0 and r.get("raw", np.nan) > 0:
            return "consistent"
        if r.get("source-centred", np.nan) > 0:
            return "style-shifted (consistent after centring)"
        return "INCONSISTENT -> expert audit"
    lsum["verdict"] = lsum.apply(verdict, axis=1)
    lsum.to_csv(TABLES / "d3_label_consistency_summary.csv")
    report("D3 - label-consistency index per class (mean over source pairs; > 0 = same name, same look)", md_table(lsum))
    LCI_SUMMARY = lsum
else:
    LCI_SUMMARY = pd.DataFrame()
tr_df = pd.DataFrame(transfer_rows)
if len(tr_df):
    tr_df.to_csv(TABLES / "d3_nearest_centroid_transfer.csv", index=False)
    report("D3 - nearest-centroid transfer between sources (shared classes only)",
           md_table(tr_df.pivot_table(index=["train_src", "test_src"], columns="space", values="macro_f1")))

cen = centroids(ZC)
keys = sorted(cen, key=lambda k: (k[1], k[0]))
if keys:
    Mx = np.array([[cen[a] @ cen[b] for b in keys] for a in keys])
    fig, ax = plt.subplots(figsize=(0.42 * len(keys) + 3, 0.42 * len(keys) + 2))
    im = ax.imshow(Mx, cmap="RdBu_r", vmin=-1, vmax=1)
    lab_ = [f"{c} | {s}" for s, c in keys]
    ax.set_xticks(range(len(keys))); ax.set_xticklabels(lab_, rotation=90, fontsize=7)
    ax.set_yticks(range(len(keys))); ax.set_yticklabels(lab_, fontsize=7)
    plt.colorbar(im, fraction=0.03); ax.set_title(f"cosine between (class | source) centroids, {MAIN_FM}, source-centred", fontsize=9)
    plt.tight_layout(); plt.savefig(FIGS / "d3_centroid_similarity.png", dpi=120); plt.show()

## 9. SERA heads on frozen tile features (RQ2)
For every split, the frozen tile features are standardized on the training part. Then:
- **Class-conditional source erasure (CCE)**: a LEACE eraser is fitted on the *training* tiles with source = the three training datasets and the class labels for conditioning. It is applied unchanged to validation and test tiles, so it needs no target data.
- **Gated-attention MIL** (Ilse et al. 2018) over the 14 views with a learned scale embedding and tile dropout. The head can use the zoomed tiles when the lesion is small (whole-plant photos) and the global view when the image is already a close-up.
- Class × source balanced sampling (as v1), AdamW, cosine schedule, **no label smoothing** (v1: label smoothing + temperature scaling made held-out ECE worse), weight decay ∈ {1e-4, 1e-2} and the best epoch chosen on the **source validation split only**.

| model | tiles | pooling | erasure | role |
|---|---|---|---|---|
| `LP-global` | global | – | – | = v1 frozen-FM linear probe (baseline) |
| `LP-global+FMM` | global | – | – | + transductive feature moment matching (best simple adaptation in Xiang et al. 2026) |
| `LP-mean` | 14 | mean | – | multi-scale information alone |
| `MIL` | 14 | gated attention | – | + learned scale selection |
| `LP-global+CCE` | global | – | class-cond. | erasure alone |
| `MIL+GE` | 14 | gated attention | global LEACE | erasure ignoring the class mix (ablation) |
| **`SERA`** | 14 | gated attention | class-cond. | **proposed** |
| `SERA-5` | 5 | gated attention | class-cond. | phone budget (global + 2×2) |

Each head trains in seconds, so every LODO model gets **10 seeds**, and the gate models also get full leave-one-source-out runs.

In [ ]:
HEAD_VARIANTS = {
    "LP-global":     dict(pool="global"),
    "LP-global+FMM": dict(pool="global", fmm=True),
    "LP-mean":       dict(pool="mean"),
    "MIL":           dict(pool="mil"),
    "LP-global+CCE": dict(pool="global", erase="cc"),
    "MIL+GE":        dict(pool="mil", erase="global"),
    "SERA":          dict(pool="mil", erase="cc"),
    "SERA-5":        dict(pool="mil", erase="cc", n_tiles=1 + 4),
}
M_ = lambda v, fm: f"{v}@{fm}"
HEAD_LODO = [M_(v, MAIN_FM) for v in HEAD_VARIANTS if v != "SERA-5"]
for fm in FMS:
    if fm != MAIN_FM:
        HEAD_LODO += [M_("LP-global", fm), M_("SERA", fm)]
HEAD_LODO.append(M_("SERA-5", EDGE_FM))
HEAD_LODO = list(dict.fromkeys(HEAD_LODO))
PROPOSED, BASE_HEAD = M_("SERA", MAIN_FM), M_("LP-global", MAIN_FM)
GATE_MODELS = list(dict.fromkeys([BASE_HEAD, PROPOSED, M_("SERA", EDGE_FM)]))
CV_HEADS = [BASE_HEAD, PROPOSED]
HEAD_TAG = hashlib.md5(f"{split_sig}|{TILE_TAG}|{HEAD_EPOCHS}|{HEAD_BATCH}|{TILE_DROP}|{KNN_K}|{SMOKE}".encode()).hexdigest()[:12]

def head_key(model, proto, split, seed):
    return run_key("H_" + model.replace("@", "_at_"), proto, split, seed)

def head_done(key):
    f = HEADS / f"{key}.json"
    if not f.exists():
        return False
    try:
        return json.loads(f.read_text()).get("tag") == HEAD_TAG
    except Exception:
        return False

class PoolHead(nn.Module):
    def __init__(self, D, K, pool, n_tiles, hidden=128):
        super().__init__()
        self.pool = pool
        self.drop = nn.Dropout(0.1)
        self.cls = nn.Linear(D, K)
        if pool == "mil":
            self.scale_emb = nn.Parameter(torch.zeros(len(TILE_GRIDS), D))
            self.register_buffer("sidx", SCALE_IDX[:n_tiles].clone())
            self.V, self.U, self.w = nn.Linear(D, hidden), nn.Linear(D, hidden), nn.Linear(hidden, 1)

    def forward(self, x, keep=None):
        """x: B x T x D standardized (and possibly erased) tile features; keep: B x T bool tile mask."""
        a = None
        if self.pool == "global":
            z = x[:, 0]
        elif self.pool == "mean":
            k = torch.ones(x.shape[:2], device=x.device) if keep is None else keep.float()
            z = (x * k.unsqueeze(-1)).sum(1) / k.sum(1, keepdim=True)
        else:
            h = x + self.scale_emb[self.sidx]
            s = self.w(torch.tanh(self.V(h)) * torch.sigmoid(self.U(h))).squeeze(-1)
            if keep is not None:
                s = s.masked_fill(~keep, -1e4)
            a = s.float().softmax(1)
            z = (a.unsqueeze(-1) * h).sum(1)
        return self.cls(self.drop(z)), a, z

def bag(fm, ix, n_tiles, ext=False):
    arr = EXT_FEATS[fm] if ext else TILE_FEATS[fm]
    ix = np.asarray(ix); o = np.argsort(ix)
    out = np.empty((len(ix), n_tiles, arr.shape[2]), np.float32)
    if len(ix):
        out[o] = arr[ix[o], :n_tiles]                 # sorted reads from the memory map
    return torch.from_numpy(out).to(DEVICE)

@torch.no_grad()
def head_forward(model, X, bs=2048, keep=None):
    model.eval(); L, A, Z = [], [], []
    for i in range(0, len(X), bs):
        l, a, z = model(X[i:i + bs], None if keep is None else keep[i:i + bs])
        L.append(l.float().cpu()); Z.append(z.float()); A.append(None if a is None else a.cpu())
    if not L:
        return np.zeros((0, model.cls.out_features), np.float32), None, torch.zeros(0, model.cls.in_features, device=DEVICE)
    return torch.cat(L).numpy(), (torch.cat(A).numpy() if A[0] is not None else None), torch.cat(Z)

def knn_dist(ref, q, k=KNN_K):
    """Mean cosine distance to the k nearest TRAINING embeddings (Sun et al., ICML 2022)."""
    ref = F.normalize(ref, dim=1); q = F.normalize(q, dim=1); out = []
    for i in range(0, len(q), 2048):
        out.append(1 - (q[i:i + 2048] @ ref.T).topk(min(k, len(ref)), dim=1).values.mean(1))
    return torch.cat(out).cpu().numpy() if out else np.zeros(0, np.float32)

def fit_head(Xtr, ytr, w, Xva, yva, K, pool, seed, wd):
    set_seed(seed)
    model = PoolHead(Xtr.shape[-1], K, pool, Xtr.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=wd)
    steps = max(1, math.ceil(len(Xtr) / HEAD_BATCH)); total = HEAD_EPOCHS * steps
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    best_f1, best_state, labels = -1.0, None, np.unique(yva)
    for ep in range(HEAD_EPOCHS):
        model.train()
        draw = torch.multinomial(w, steps * HEAD_BATCH, replacement=True).to(DEVICE)
        for s in range(steps):
            b = draw[s * HEAD_BATCH:(s + 1) * HEAD_BATCH]
            x, keep = Xtr[b], None
            if pool != "global" and TILE_DROP > 0:
                keep = torch.rand(x.shape[:2], device=DEVICE) > TILE_DROP
                keep[:, 0] |= ~keep.any(1)
            loss = F.cross_entropy(model(x, keep)[0], ytr[b])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if ep % 2 == 1 or ep == HEAD_EPOCHS - 1:
            f1 = f1_macro_np(yva, head_forward(model, Xva)[0].argmax(1), labels)
            if f1 > best_f1:
                best_f1, best_state = f1, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state); model.eval()
    return model, best_f1

def build_and_train(model_name, proto, split, seed, keep_X=False):
    variant, fm = model_name.split("@"); spec = HEAD_VARIANTS[variant]
    S = SPLITS[(proto, split)]; ext = proto == "ext"
    tr, va, te = S["train"], S["val"], S["test"]
    classes = np.array(sorted(set(Y_ALL[tr].tolist())))
    K = len(classes)
    g2l = torch.full((len(CLASSES),), -1, dtype=torch.long); g2l[torch.tensor(classes)] = torch.arange(K)
    nt = 1 if spec["pool"] == "global" else spec.get("n_tiles", N_TILES)
    Xtr, Xva, Xte = bag(fm, tr, nt), bag(fm, va, nt), bag(fm, te, nt, ext)
    mu, sd = Xtr.mean((0, 1)), Xtr.std((0, 1)) + 1e-6
    Xtr, Xva, Xte = (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd
    info = dict(erase_rank=0)
    if spec.get("erase"):
        er = Eraser(Xtr.flatten(0, 1), D_ALL[tr].repeat_interleave(nt).to(DEVICE),
                    Y_ALL[tr].repeat_interleave(nt).to(DEVICE) if spec["erase"] == "cc" else None)
        Xtr, Xva, Xte = er(Xtr), er(Xva), er(Xte)
        info["erase_rank"] = er.rank
        del er
    if spec.get("fmm") and len(Xte):     # transductive: match the unlabelled test set's moments to training (mean 0, sd 1)
        Xte = (Xte - Xte.mean((0, 1))) / (Xte.std((0, 1)) + 1e-6)
    ytr = g2l[Y_ALL[tr]].to(DEVICE); yva = g2l[Y_ALL[va]].numpy()
    cls_n = Counter(ytr.tolist()); dom = D_ALL[tr]; dom_n = Counter(dom.tolist())
    w = torch.tensor([cls_n[int(c)] ** -0.5 * dom_n[int(d)] ** -0.5 for c, d in zip(ytr.tolist(), dom.tolist())],
                     dtype=torch.float)
    best = None
    for wd in (1e-4, 1e-2):
        model, f1 = fit_head(Xtr, ytr, w, Xva, yva, K, spec["pool"], seed, wd)
        if best is None or f1 > best[1]:
            best = (model, f1, wd)
    model, vf1, wd = best
    lv, _, zv = head_forward(model, Xva)
    lt, at, zt = head_forward(model, Xte)
    _, _, ztr = head_forward(model, Xtr)
    out = dict(model=model, classes=classes, val_idx=va, test_idx=te, val_logits=lv, test_logits=lt,
               val_knn=knn_dist(ztr, zv), test_knn=knn_dist(ztr, zt), test_att=at,
               info=dict(info, weight_decay=wd, val_macro_f1=vf1))
    if keep_X:
        out["X"] = (Xtr, Xva, Xte)
    else:
        del Xtr, Xva, Xte
    return out

def run_head(model_name, proto, split, seed):
    key = head_key(model_name, proto, split, seed)
    if head_done(key):
        return None
    t = time.time()
    r = build_and_train(model_name, proto, split, seed)
    extra = {"test_att": r["test_att"].astype(np.float16)} if (r["test_att"] is not None and seed == HEAD_SEEDS[0]
                                                                and proto in ("lodo", "ext")) else {}
    np.savez_compressed(HEADS / f"{key}.npz", val_idx=r["val_idx"], val_logits=r["val_logits"], test_idx=r["test_idx"],
                        test_logits=r["test_logits"], val_knn=r["val_knn"], test_knn=r["test_knn"],
                        classes=r["classes"], **extra)
    S = SPLITS[(proto, split)]
    pos = {g: i for i, g in enumerate(r["test_idx"])}
    sel = np.array([pos[g] for g in S["test_id"]], dtype=int)
    y = y_of(proto, S["test_id"])
    f1 = f1_macro_np(y, r["classes"][r["test_logits"][sel].argmax(1)], np.unique(y)) if len(sel) else np.nan
    save_json(dict(key=key, model=model_name, protocol=proto, split=split, seed=seed, classes=r["classes"].tolist(),
                   tag=HEAD_TAG, minutes=(time.time() - t) / 60, test_id_macro_f1=f1, **r["info"]), HEADS / f"{key}.json")
    del r; torch.cuda.empty_cache()
    return f1

### 9.1 Run all heads (priority order, resumable)
Order: LODO seed 0 for every head → leave-one-source-out runs of the gate models → in-domain CV → external → remaining seeds.

In [ ]:
lodo_splits = [k for k in SPLITS if k[0] == "lodo"]
inner_splits = [k for k in SPLITS if k[0] == "inner"]
cv_splits = [k for k in SPLITS if k[0] == "cv"]
HPLAN = [(m, sp, HEAD_SEEDS[0]) for m in HEAD_LODO for sp in lodo_splits]
HPLAN += [(m, sp, s) for s in INNER_SEEDS for m in GATE_MODELS for sp in inner_splits]
HPLAN += [(m, sp, HEAD_SEEDS[0]) for m in CV_HEADS for sp in cv_splits]
if ("ext", "all") in SPLITS:
    HPLAN += [(m, ("ext", "all"), s) for s in INNER_SEEDS for m in GATE_MODELS]
HPLAN += [(m, k, s) for s in INNER_SEEDS for m in GATE_MODELS for k in SPLITS if k[0] == "single"]
HPLAN += [(m, sp, s) for s in HEAD_SEEDS[1:] for m in HEAD_LODO for sp in lodo_splits]
done = sum(head_done(head_key(m, *sp, s)) for m, sp, s in HPLAN)
print(f"head plan: {len(HPLAN)} runs | already done: {done}")
t_h = time.time()
for i, (m, (proto, split), s) in enumerate(tqdm(HPLAN, desc="heads")):
    f1 = run_head(m, proto, split, s)
    if f1 is not None and (s == HEAD_SEEDS[0] and proto == "lodo"):
        print(f"{m:28s} {proto}/{split:18s} s{s}: macro-F1(ID test) = {f1:.3f}")
print(f"heads finished in {(time.time() - t_h) / 60:.1f} min | elapsed {hours():.2f} h")

## 10. Fine-tuned baselines (earlier runs reused; missing runs trained, time-guarded)
Earlier runs are reused when their split signature and training budget match (`RUN_TAG`). Missing runs are trained with **exactly v1's recipe**:
1. **first**, the gate model(s) in `FT_GATE_MODELS` on LODO and on every leave-one-source-out split (seed 0). New runs also store **kNN feature distances**, so the risk-controlled gate (section 12) can be calibrated on the strongest fine-tuned network too;
2. then the remaining seeds of `FT_MODELS`, so that every comparison has ≥ 3 seeds.

Runs stop at `TIME_BUDGET_H`; re-run with this output attached to continue.

In [ ]:
BACKBONES = {
    "resnet50":        dict(timm="resnet50.a1_in1k", lr=3e-4),
    "efficientnet_b0": dict(timm="efficientnet_b0.ra_in1k", lr=3e-4),
    "mobilenetv3_l":   dict(timm="mobilenetv3_large_100.ra_in1k", lr=3e-4),
    "convnext_tiny":   dict(timm="convnext_tiny.fb_in1k", lr=2e-4),
    "vit_small":       dict(timm="vit_small_patch16_224.augreg_in21k_ft_in1k", lr=1e-4),
}
FT_BACKBONE = {"ResNet50": "resnet50", "EfficientNetB0": "efficientnet_b0", "MobileNetV3L": "mobilenetv3_l",
               "ConvNeXtT": "convnext_tiny", "ViTS": "vit_small", "ViTS-kNN": "vit_small"}
V1_MODELS = list(dict.fromkeys(["ResNet50", "EfficientNetB0", "MobileNetV3L", "ConvNeXtT", "ViTS", "FM:DINOv2-B", "FM:DINOv3-B",
                                "FM:BioCLIP-2", "DG:MixStyle", "DG:AmpMix+Cons", "DG:SWA", "SILA-Net"] + FT_MODELS + FT_GATE_MODELS))
CACHE_T = torch.from_numpy(CACHE)

def gpu_augment(xb):
    """v1 augmentation: random-resized crop + rotation + flip (one affine warp), mild b/c/s jitter, no hue."""
    x = xb.float() / 255.0
    B, dev = x.size(0), x.device
    area = torch.empty(B, device=dev).uniform_(0.55, 1.0)
    logr = torch.empty(B, device=dev).uniform_(math.log(3 / 4), math.log(4 / 3))
    sw = torch.sqrt(area * torch.exp(logr)).clamp(max=1.0); sh = torch.sqrt(area / torch.exp(logr)).clamp(max=1.0)
    tx = (1 - sw) * torch.empty(B, device=dev).uniform_(-1, 1); ty = (1 - sh) * torch.empty(B, device=dev).uniform_(-1, 1)
    ang = torch.empty(B, device=dev).uniform_(-15, 15) * math.pi / 180
    flip = torch.where(torch.rand(B, device=dev) < 0.5, -1.0, 1.0)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.stack([torch.stack([cos * sw * flip, -sin * sh, tx], 1), torch.stack([sin * sw * flip, cos * sh, ty], 1)], 1)
    grid = F.affine_grid(theta, (B, 3, IMG_SIZE, IMG_SIZE), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    b = torch.empty(B, 1, 1, 1, device=dev).uniform_(0.8, 1.2); c = torch.empty(B, 1, 1, 1, device=dev).uniform_(0.8, 1.2)
    s = torch.empty(B, 1, 1, 1, device=dev).uniform_(0.8, 1.2)
    x = x * b
    gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
    x = (x - gray.mean((2, 3), keepdim=True)) * c + gray.mean((2, 3), keepdim=True)
    gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
    return (gray + (x - gray) * s).clamp(0, 1)

def eval_view(xb):
    return F.interpolate(xb.float() / 255.0, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", antialias=True, align_corners=False)

class FTNet(nn.Module):
    def __init__(self, backbone, n_classes, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(BACKBONES[backbone]["timm"], pretrained=pretrained and PRETRAINED,
                                          num_classes=n_classes, drop_rate=0.2)
        cfg = timm.data.resolve_data_config({}, model=self.backbone)
        self.register_buffer("mean", torch.tensor(cfg["mean"]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(cfg["std"]).view(1, 3, 1, 1))
    def forward(self, x01):
        return self.backbone((x01 - self.mean) / self.std)

@torch.no_grad()
def predict_logits(model, ix, bs=64):
    model.eval(); out = []
    for b in batches(np.asarray(ix), bs):
        x = eval_view(CACHE_T[b].to(DEVICE, non_blocking=True))
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            lg = model(x)
        if not torch.isfinite(lg).all():
            lg = model(x)
        out.append(lg.float().cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, 0), np.float32)

def train_ft(backbone, tr, va, classes, seed):
    """v1 training recipe (timm head): class x source sampler, AdamW (head lr x10), warm-up + cosine,
    label smoothing 0.1, AMP, best validation macro-F1 checkpoint."""
    set_seed(seed)
    K = len(classes)
    g2l = torch.full((len(CLASSES),), -1, dtype=torch.long); g2l[torch.tensor(classes)] = torch.arange(K)
    y_tr = g2l[Y_ALL[tr]]; y_va = g2l[Y_ALL[va]].numpy()
    model = FTNet(backbone, K).to(DEVICE)
    hp = {id(p) for p in model.backbone.get_classifier().parameters()}
    base_lr = BACKBONES[backbone]["lr"]
    opt = torch.optim.AdamW([{"params": [p for p in model.parameters() if id(p) not in hp], "lr": base_lr},
                             {"params": [p for p in model.parameters() if id(p) in hp], "lr": base_lr * 10}], weight_decay=0.05)
    steps = max(1, math.ceil(len(tr) / BATCH)); total = EPOCHS * steps
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / steps) *
                                              (0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total)))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    cls_n = Counter(y_tr.tolist()); dom = D_ALL[tr]; dom_n = Counter(dom.tolist())
    w = torch.tensor([cls_n[int(c)] ** -0.5 * dom_n[int(d)] ** -0.5 for c, d in zip(y_tr, dom)], dtype=torch.float)
    tr_t = torch.as_tensor(tr)
    best, best_state, hist = -1.0, None, []
    for ep in range(EPOCHS):
        model.train(); run_loss = 0.0
        draw = torch.multinomial(w, steps * BATCH, replacement=True)
        for s in range(steps):
            sel = draw[s * BATCH:(s + 1) * BATCH]
            x = gpu_augment(CACHE_T[tr_t[sel]].to(DEVICE, non_blocking=True)); yb = y_tr[sel].to(DEVICE)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                loss = F.cross_entropy(model(x).float(), yb, label_smoothing=0.1)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step(); run_loss += loss.item()
        f1 = f1_score(y_va, predict_logits(model, va).argmax(1), average="macro")
        hist.append(dict(epoch=ep, loss=run_loss / steps, val_macro_f1=f1))
        if f1 > best:
            best, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, hist

@torch.no_grad()
def ft_embed(model, ix, bs=128):
    """Penultimate (pre-logit) features of a fine-tuned model, evaluation view."""
    model.eval(); out = []
    for b in batches(np.asarray(ix), bs):
        x = (eval_view(CACHE_T[b].to(DEVICE, non_blocking=True)) - model.mean) / model.std
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            f = model.backbone.forward_head(model.backbone.forward_features(x), pre_logits=True)
        out.append(f.float())
    return torch.cat(out)

# priority: gate model on LODO + every leave-one-source-out split (seed 0), then the remaining baseline seeds
FT_PLAN = []
if RUN_FT_BASELINES:
    FT_PLAN += [(m, ("lodo", h), SEEDS[0]) for m in FT_GATE_MODELS for h in dss]
    FT_PLAN += [(m, sp, SEEDS[0]) for m in FT_GATE_MODELS for sp in inner_splits]
    FT_PLAN += [(m, ("lodo", h), s) for s in SEEDS for m in FT_MODELS for h in dss]
FT_PLAN = list(dict.fromkeys(FT_PLAN))
todo = [p for p in FT_PLAN if not run_done(run_key(p[0], *p[1], p[2]))]
print(f"fine-tuned plan: {len(FT_PLAN)} runs | to train: {len(todo)}")
for m, (proto, split), seed in todo:
    if hours() > TIME_BUDGET_H:
        print(f"time budget reached ({hours():.1f} h): remaining fine-tuning runs skipped; re-run with this output attached")
        break
    S = SPLITS[(proto, split)]; key = run_key(m, proto, split, seed)
    classes = sorted(set(int(c) for c in Y_ALL[S["train"]].tolist()))
    t = time.time()
    model, hist = train_ft(FT_BACKBONE[m], S["train"], S["val"], classes, seed)
    va_logits, te_logits = predict_logits(model, S["val"]), predict_logits(model, S["test"])
    ref = np.random.default_rng(seed).permutation(S["train"])[:6000]          # kNN reference: training images
    z_ref = ft_embed(model, ref)
    val_knn, test_knn = knn_dist(z_ref, ft_embed(model, S["val"])), knn_dist(z_ref, ft_embed(model, S["test"]))
    np.savez_compressed(RUNS / f"{key}.npz", val_idx=S["val"], val_logits=va_logits, test_idx=S["test"],
                        test_logits=te_logits, classes=np.array(classes), val_knn=val_knn, test_knn=test_knn)
    pos = {g: i for i, g in enumerate(S["test"])}
    y_te = Y_ALL[S["test_id"]].numpy(); pid = np.array(classes)[te_logits[[pos[g] for g in S["test_id"]]].argmax(1)]
    f1 = f1_score(y_te, pid, labels=sorted(set(y_te)), average="macro") if len(y_te) else float("nan")
    save_json(dict(key=key, model=m, spec=dict(backbone=FT_BACKBONE[m], head="timm"), protocol=proto, split=split,
                   seed=seed, classes=classes, epochs=EPOCHS, history=hist, minutes=(time.time() - t) / 60,
                   test_id_macro_f1=f1, params_m=count_params(model), run_tag=RUN_TAG), RUNS / f"{key}.json")
    print(f"{key}: macro-F1(ID test)={f1:.3f} | {(time.time() - t) / 60:.1f} min | elapsed {hours():.2f} h")
    del model; torch.cuda.empty_cache()

## 11. Results (RQ1, RQ2): LODO macro-F1, ablation, encoders, degradation, statistics
- Macro-F1 on each held-out source's images of classes seen in training; mean ± sd over seeds.
- **Statistics**: hierarchical bootstrap of the mean (over held-out sets) difference in macro-F1. Photo groups are resampled within each held-out set and seeds within each model (1000 replicates). This respects both sources of variance. Wilcoxon over (held-out × seed) pairs is shown as a secondary test, Holm-corrected.

In [ ]:
_RUN_CACHE = {}

def load_any(model, proto, split, seed):
    """Load a run of a v2 head (name contains '@') or a v1-format fine-tuned run. None if missing/stale."""
    k = (model, proto, split, seed)
    if k in _RUN_CACHE:
        return _RUN_CACHE[k]
    if "@" in model:
        key = head_key(model, proto, split, seed); f = HEADS / f"{key}.npz"; ok = head_done(key)
    else:
        key = run_key(model, proto, split, seed); f = RUNS / f"{key}.npz"; ok = run_done(key)
    r = None
    if ok and f.exists():
        z = np.load(f); r = {kk: z[kk] for kk in z.files}; r["classes"] = np.asarray(r["classes"])
    _RUN_CACHE[k] = r
    return r

def eval_test(r, proto, split):
    S = SPLITS[(proto, split)]
    pos = {g: i for i, g in enumerate(r["test_idx"])}
    sel = np.array([pos[g] for g in S["test_id"]], dtype=int)
    y = y_of(proto, S["test_id"]); pred = r["classes"][r["test_logits"][sel].argmax(1)]
    labels = np.unique(y)
    pc = f1_per_class_np(y, pred, labels) if len(labels) else []
    groups = idx.group.values[S["test_id"]] if proto != "ext" else np.arange(len(y)).astype(str)
    return dict(macro_f1=float(np.mean(pc)) if len(pc) else np.nan, acc=float((pred == y).mean()) if len(y) else np.nan,
                per_class={CLASSES[c]: v for c, v in zip(labels, pc)}, y=y, pred=pred, groups=groups)

ALL_SEEDS = sorted(set(HEAD_SEEDS) | set(SEEDS) | {0, 1, 2})
EV = {}
for m in HEAD_LODO + V1_MODELS:
    for h in dss:
        for s in ALL_SEEDS:
            r = load_any(m, "lodo", h, s)
            if r is not None:
                EV[(m, h, s)] = eval_test(r, "lodo", h)
V1_AVAIL = [m for m in V1_MODELS if any(k[0] == m for k in EV)]
print("v2 heads with results:", len({k[0] for k in EV if '@' in k[0]}), "| v1/fine-tuned models:", V1_AVAIL)

def lodo_table(models):
    rows = []
    for m in models:
        for h in dss:
            v = [EV[k]["macro_f1"] for k in EV if k[0] == m and k[1] == h]
            if v:
                rows.append(dict(model=m, held_out=h, n_seeds=len(v), macro_f1=np.mean(v), sd=np.std(v)))
    return pd.DataFrame(rows)

def show_table(models, title, fname, order=None):
    lt = lodo_table(models)
    if not len(lt):
        return lt
    piv = lt.pivot(index="model", columns="held_out", values="macro_f1")
    sd_ = lt.pivot(index="model", columns="held_out", values="sd")
    cell = piv.round(3).astype(str) + " ± " + sd_.round(3).astype(str)
    cell["mean"] = piv.mean(axis=1).round(3)
    cell["seeds"] = lt.groupby("model").n_seeds.min()
    cell = cell.reindex(order) if order else cell.sort_values("mean", ascending=False)
    cell = cell.dropna(how="all")
    cell.to_csv(TABLES / fname)
    report(title, md_table(cell))
    return lt

LT_ALL = show_table(HEAD_LODO + V1_AVAIL, "RQ2 - LODO macro-F1 on held-out ID classes, all models (mean ± sd over seeds)",
                    "lodo_macro_f1_all.csv")
show_table([M_(v, MAIN_FM) for v in HEAD_VARIANTS if v != "SERA-5"],
           f"RQ2 - SERA ablation on {MAIN_FM} (LODO macro-F1)", "lodo_ablation.csv",
           order=[M_(v, MAIN_FM) for v in HEAD_VARIANTS if v != "SERA-5"])
show_table([M_(v, fm) for fm in FMS for v in ("LP-global", "SERA", "SERA-5")],
           "RQ2 - does SERA help every encoder? (LODO macro-F1)", "lodo_by_encoder.csv")

In [ ]:
# ---- in-domain CV -> degradation (RQ1)
def cv_indomain(model):
    gi, gp = [], []
    for k in range(CV_FOLDS):
        r = load_any(model, "cv", f"fold{k}", 0)
        if r is None:
            return None
        gi.append(r["test_idx"]); gp.append(r["classes"][r["test_logits"].argmax(1)])
    gi, gp = np.concatenate(gi), np.concatenate(gp)
    out = {}
    for held in dss:
        shared = set(Y_ALL[SPLITS[("lodo", held)]["test_id"]].tolist())
        msk = (idx.dataset.values[gi] == held) & np.isin(Y_ALL.numpy()[gi], list(shared))
        if msk.any():
            out[held] = f1_macro_np(Y_ALL.numpy()[gi][msk], gp[msk], np.array(sorted(shared)))
    return out

deg_rows = []
for m in CV_HEADS + V1_AVAIL:
    ind = cv_indomain(m)
    if not ind:
        continue
    for h, v in ind.items():
        lo = [EV[k]["macro_f1"] for k in EV if k[0] == m and k[1] == h]
        if lo:
            deg_rows.append(dict(model=m, held_out=h, in_domain=v, lodo=np.mean(lo), drop_points=100 * (v - np.mean(lo))))
deg = pd.DataFrame(deg_rows)
if len(deg):
    deg.to_csv(TABLES / "degradation.csv", index=False)
    dp = deg.pivot(index="model", columns="held_out", values="drop_points"); dp["mean"] = dp.mean(axis=1)
    report("RQ1 - in-domain minus LODO macro-F1 (percentage points)", md_table(dp.sort_values("mean"), "{:.1f}"))

In [ ]:
# ---- statistics: proposed vs every other model
def hier_boot(A, B, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    pre = []
    for h in dss:
        ea = [EV[k] for k in sorted(EV) if k[0] == A and k[1] == h]
        eb = [EV[k] for k in sorted(EV) if k[0] == B and k[1] == h]
        if not ea or not eb:
            continue
        y = ea[0]["y"]; labels = np.unique(y)
        _, ginv = np.unique(ea[0]["groups"], return_inverse=True)
        members = [np.where(ginv == g)[0] for g in range(ginv.max() + 1)]
        pre.append((y, labels, members, [e["pred"] for e in ea], [e["pred"] for e in eb]))
    if not pre:
        return None
    point = np.mean([np.mean([f1_macro_np(y, p, l) for p in pa]) - np.mean([f1_macro_np(y, p, l) for p in pb])
                     for y, l, _, pa, pb in pre])
    D = np.empty(n_boot)
    for b in range(n_boot):
        diffs = []
        for y, l, members, pa, pb in pre:
            ix = np.concatenate([members[g] for g in rng.integers(0, len(members), len(members))])
            fa = np.mean([f1_macro_np(y[ix], pa[i][ix], l) for i in rng.integers(0, len(pa), len(pa))])
            fb = np.mean([f1_macro_np(y[ix], pb[i][ix], l) for i in rng.integers(0, len(pb), len(pb))])
            diffs.append(fa - fb)
        D[b] = np.mean(diffs)
    lo, hi = np.percentile(D, [2.5, 97.5])
    return point, lo, hi, min(1.0, 2 * min((D <= 0).mean(), (D >= 0).mean())), len(pre)

stat_rows = []
for B in [m for m in HEAD_LODO + V1_AVAIL if m != PROPOSED]:
    res = hier_boot(PROPOSED, B, n_boot=100 if SMOKE else 1000)
    if res is None:
        continue
    point, lo, hi, p_b, nh = res
    pairs = [(EV[(PROPOSED, h, s)]["macro_f1"], EV[(B, h, s)]["macro_f1"]) for h in dss for s in ALL_SEEDS
             if (PROPOSED, h, s) in EV and (B, h, s) in EV]
    try:
        p_w = stats.wilcoxon(*np.array(pairs).T).pvalue if len(pairs) >= 6 and np.any(np.diff(np.array(pairs), axis=1)) else np.nan
    except ValueError:
        p_w = np.nan
    stat_rows.append(dict(baseline=B, diff_macro_f1=point, ci_low=lo, ci_high=hi, p_bootstrap=p_b,
                          wilcoxon_p=p_w, paired_runs=len(pairs), held_out_sets=nh))
STAT = pd.DataFrame(stat_rows)
if len(STAT):
    STAT["p_bootstrap_holm"] = holm(STAT.p_bootstrap.values)
    STAT["significant"] = (STAT.ci_low > 0) | (STAT.ci_high < 0)
    STAT = STAT.sort_values("diff_macro_f1").reset_index(drop=True)
    STAT.to_csv(TABLES / "stats_proposed_vs_all.csv", index=False)
    report(f"Statistics - {PROPOSED} minus each model (hierarchical bootstrap over photo groups and seeds)", md_table(STAT, "{:.4f}"))

# per-class F1 for the focus models
best_ft = (max([m for m in V1_AVAIL if m in FT_BACKBONE], key=lambda m: lodo_table([m]).macro_f1.mean())
           if any(m in FT_BACKBONE for m in V1_AVAIL) else None)
FOCUS = [m for m in [PROPOSED, M_("MIL", MAIN_FM), M_("LP-global+CCE", MAIN_FM), BASE_HEAD, best_ft, "SILA-Net"] if m]
pc_rows = [dict(model=m, held_out=h, seed=s, **{f"F1_{c}": v for c, v in ev["per_class"].items()})
           for (m, h, s), ev in EV.items() if m in FOCUS]
if pc_rows:
    pc = pd.DataFrame(pc_rows).groupby(["model", "held_out"]).mean(numeric_only=True).drop(columns="seed")
    pc.to_csv(TABLES / "per_class_f1_focus.csv")
    report("Per-class F1 on held-out sets (focus models, mean over seeds)", md_table(pc))

## 12. Risk-controlled treatment gate (RQ3)
A remedy group is **issued** when the gate accepts the image; otherwise the app says *"uncertain: re-take a close, well-lit photo or ask the extension officer"*. The controlled quantity is the **wrong-remedy risk** R = P(issue ∧ issued group ≠ true group) over *all* images of a new source, including diseases the model never saw. An unseen disease whose treatment group is known (e.g. botrytis → fungal foliar) counts as correct when that group is issued.

**Risk scores** (higher = riskier), each converted to validation-quantile units *u* = F_val(score) so that thresholds transfer between models:
- `MSP` = 1 − max disease probability; `Remedy` = 1 − max remedy-group probability (temperature-scaled on source validation);
- `kNN` = mean cosine distance to the 10 nearest training embeddings (catches confident errors far from the training data);
- `Remedy+kNN` = max(u_Remedy, u_kNN): withhold if *either* signal is unusual.

**Calibration of the threshold λ (accept iff u ≤ λ)** by conformal risk control (Angelopoulos et al., ICLR 2024) with loss 1[u ≤ λ ∧ wrong]:
- `val-CRC`: on the pooled source validation split (assumes the test source looks like training; v1 shows it does not);
- **`LOSO-CRC` (ours)**: on the *out-of-source* predictions of the leave-one-source-out models (each training source scored by a model that never saw it). The finite-sample-corrected risk is averaged over sources. This targets the risk on a *new* source under source-level exchangeability (Dunn et al., *JASA* 2023), approximate with 3 training sources;
- `LOSO-CRC-worst`: the same with the maximum over sources (conservative).
- For continuity with v1: `CP-remedy(val)`, i.e. issue only if the 90% conformal remedy set is a singleton.

Reported per model × score × calibration, averaged with **equal weight over held-out sets and seeds**: acceptance, **risk** (target ε = 5%), **violation rate** (share of held-out × seed runs with risk > ε), wrong rate among issued remedies, unseen diseases withheld. Separately, *ranking quality* of each score: AURC and wrong-remedy rate among issued at **matched acceptance** (30%, 50%), and AUROC for unseen vs seen diseases.

In [ ]:
from sklearn.metrics import roc_auc_score

def decision_view(r, proto):
    classes = r["classes"]
    g2l = {g: i for i, g in enumerate(classes)}
    yv = Y_ALL[r["val_idx"]].numpy(); yv_l = np.array([g2l[g] for g in yv])
    T = fit_temperature(r["val_logits"], yv_l)
    pv, pt = softmax(r["val_logits"], T), softmax(r["test_logits"], T)
    Pv, Pt = group_probs(pv, classes), group_probs(pt, classes)
    gv, gt = CLASS_GROUP[yv], g_of(proto, r["test_idx"])
    raw = {"MSP": (1 - pv.max(1), 1 - pt.max(1)), "Remedy": (1 - Pv.max(1), 1 - Pt.max(1))}
    if "val_knn" in r:
        raw["kNN"] = (r["val_knn"], r["test_knn"])
    U = {k: (cdf_of(v, v), cdf_of(v, t)) for k, (v, t) in raw.items()}
    if "kNN" in U:
        U["Remedy+kNN"] = (np.maximum(U["Remedy"][0], U["kNN"][0]), np.maximum(U["Remedy"][1], U["kNN"][1]))
    qr = conf_quantile(1 - Pv[np.arange(len(gv)), gv], ALPHA)
    rset = (1 - Pt) <= qr
    yt = y_of(proto, r["test_idx"])
    return dict(T=T, U=U, wrong_v=Pv.argmax(1) != gv, wrong_t=Pt.argmax(1) != gt, issued_t=Pt.argmax(1), true_g=gt,
                cp_accept=rset.sum(1) == 1, cp_wrong=(rset.sum(1) == 1) & (rset.argmax(1) != gt),
                unseen=~np.isin(yt, classes), pt=pt, classes=classes, y=yt)

def crc_lambda(u_list, wrong_list, eps=RISK_EPS, mode="mean"):
    """Conformal risk control for L(lam) = 1[u <= lam and wrong] (monotone in lam). Per calibration source the
    finite-sample bound is n/(n+1) * R_n(lam) + 1/(n+1); sources are averaged (mode='mean') or the worst is used
    ('max'). Returns the largest lam meeting eps, or -1 (accept nothing)."""
    grid = np.unique(np.concatenate([[0.0, 1.0]] + [np.asarray(u) for u in u_list]))
    risks = []
    for u, w in zip(u_list, wrong_list):
        n = len(u)
        if n == 0:
            continue
        uw = np.sort(np.asarray(u)[np.asarray(w)])
        risks.append(n / (n + 1) * np.searchsorted(uw, grid, side="right") / n + 1 / (n + 1))
    if not risks:
        return -1.0
    agg = np.mean(risks, 0) if mode == "mean" else np.max(risks, 0)
    ok = np.where(agg <= eps)[0]
    return float(grid[ok[-1]]) if len(ok) else -1.0

def calib_views(model, proto, split, seed):
    """Out-of-source decision views used by LOSO-CRC: inner models (LODO) or the LODO models themselves (external)."""
    if proto == "lodo":
        cal = [("inner", f"{split}|{s}") for s in dss if s != split]
    elif proto == "ext":
        cal = [("lodo", s) for s in dss]
    else:
        return None
    views = []
    for p2, sp2 in cal:
        r = load_any(model, p2, sp2, seed)
        if r is None:
            return None
        views.append(decision_view(r, p2))
    return views

def gate_rows(model, proto, split, seed, require_loso=False):
    """All calibrations of one run. require_loso=True skips runs without out-of-source calibration data, so that
    every calibration of a model is averaged over exactly the same runs."""
    r = load_any(model, proto, split, seed)
    if r is None:
        return [], []
    lv = calib_views(model, proto, split, seed)
    if require_loso and not lv:
        return [], []
    dv = decision_view(r, proto)
    S = SPLITS[(proto, split)]
    idm = np.isin(r["test_idx"], S["test_id"])
    base = dict(model=model, proto=proto, held_out=split, seed=seed)
    rows, rank = [], []
    def add(score, cal, accept, lam=np.nan):
        wrong = accept & dv["wrong_t"]
        rows.append(dict(base, score=score, calibration=cal, lam=lam, accept=accept.mean(),
                         accept_id=accept[idm].mean() if idm.any() else np.nan,
                         risk=wrong.mean(), violation=float(wrong.mean() > RISK_EPS),
                         wrong_among_issued=wrong.sum() / max(1, accept.sum()),
                         withheld_unseen=(~accept[dv["unseen"]]).mean() if dv["unseen"].any() else np.nan,
                         n=len(accept)))
    add("-", "no gate", np.ones(len(dv["wrong_t"]), bool))
    add("Remedy-set", "CP-remedy(val)", dv["cp_accept"])
    for score, (uv, ut) in dv["U"].items():
        lam = crc_lambda([uv], [dv["wrong_v"]]); add(score, "val-CRC", ut <= lam, lam)
        if lv:
            for mode, name in (("mean", "LOSO-CRC"), ("max", "LOSO-CRC-worst")):
                lam = crc_lambda([v["U"][score][1] for v in lv], [v["wrong_t"] for v in lv], mode=mode)
                add(score, name, ut <= lam, lam)
        rr = dict(base, score=score, aurc=aurc(-ut, ~dv["wrong_t"]),
                  auroc_unseen=roc_auc_score(dv["unseen"], ut) if 0 < dv["unseen"].sum() < len(ut) else np.nan)
        for a in MATCHED_ACCEPT:
            thr = np.quantile(ut, a); acc = ut <= thr
            rr[f"wrong_among_issued@{int(a * 100)}%"] = (acc & dv["wrong_t"]).sum() / max(1, acc.sum())
        rank.append(rr)
    return rows, rank

GATE_FOCUS = list(dict.fromkeys(GATE_MODELS + [m for m in FT_GATE_MODELS if m in V1_AVAIL] +
                                [M_("MIL", MAIN_FM), M_("LP-global+CCE", MAIN_FM)] +
                                [m for m in ["EfficientNetB0", "ConvNeXtT", "ViTS", "SILA-Net", "FM:DINOv2-B", "FM:DINOv3-B"]
                                 if m in V1_AVAIL]))
G_ROWS, R_ROWS = [], []
for m in tqdm(GATE_FOCUS, desc="gates"):
    has_loso = any(calib_views(m, "lodo", h, s) for h in dss for s in ALL_SEEDS if load_any(m, "lodo", h, s) is not None)
    for h in dss:
        for s in ALL_SEEDS:
            a, b = gate_rows(m, "lodo", h, s, require_loso=has_loso); G_ROWS += a; R_ROWS += b
GT, RK = pd.DataFrame(G_ROWS), pd.DataFrame(R_ROWS)
if len(GT):
    GT.to_csv(TABLES / "gate_all_runs.csv", index=False); RK.to_csv(TABLES / "gate_score_ranking_all_runs.csv", index=False)
    GAGG = (GT.groupby(["model", "score", "calibration"])
              .agg(accept=("accept", "mean"), accept_id=("accept_id", "mean"), risk=("risk", "mean"),
                   violation_rate=("violation", "mean"), wrong_among_issued=("wrong_among_issued", "mean"),
                   withheld_unseen=("withheld_unseen", "mean"), runs=("risk", "size")))
    GAGG.to_csv(TABLES / "gate_summary.csv")
    report(f"RQ3 - gate summary for {PROPOSED} (target risk eps = {RISK_EPS:.0%}; mean over held-out sets and seeds)",
           md_table(GAGG.loc[PROPOSED] if PROPOSED in GAGG.index.get_level_values(0) else GAGG.head(0), "{:.3f}"))
    report("RQ3 - gate summary, all focus models", md_table(GAGG, "{:.3f}"))
    RAGG = RK.groupby(["model", "score"]).mean(numeric_only=True).drop(columns=["seed"], errors="ignore")
    RAGG.to_csv(TABLES / "gate_score_ranking.csv")
    report("RQ3 - ranking quality of the risk scores (lower AURC / wrong-at-matched-acceptance is better)", md_table(RAGG, "{:.3f}"))

In [ ]:
# headline + figures
if len(GT) and PROPOSED in set(GT.model):
    g = GAGG.loc[PROPOSED]
    def row_(score, cal):
        return g.loc[(score, cal)] if (score, cal) in g.index else None
    lines = []
    ng = row_("-", "no gate")
    if ng is not None:
        lines.append(f"- no gate: wrong-remedy risk **{ng.risk:.1%}** of all held-out images")
    for score, cal in [("MSP", "val-CRC"), ("Remedy", "val-CRC"), ("Remedy-set", "CP-remedy(val)"),
                       ("Remedy", "LOSO-CRC"), ("Remedy+kNN", "LOSO-CRC"), ("Remedy+kNN", "LOSO-CRC-worst")]:
        r_ = row_(score, cal)
        if r_ is not None:
            lines.append(f"- {score} / {cal}: risk {r_.risk:.1%} (violations {r_.violation_rate:.0%} of runs), "
                         f"accepts {r_.accept:.0%} of images, {r_.wrong_among_issued:.0%} of issued remedies wrong, "
                         f"withholds {r_.withheld_unseen:.0%} of unseen diseases")
    if CROP == "onion":
        lines.append("- v1 reference (SILA-Net, cross-source conformal level): accepted 11%, 23% of issued remedies wrong")
    report(f"RQ3 headline ({PROPOSED})", "\n".join(lines))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    mk = {"val-CRC": "o", "LOSO-CRC": "*", "LOSO-CRC-worst": "P", "CP-remedy(val)": "s", "no gate": "x"}
    for (score, cal), r_ in g.iterrows():
        axes[0].scatter(r_.accept, r_.risk, marker=mk.get(cal, "o"), s=140 if cal == "LOSO-CRC" else 60, label=f"{score} / {cal}")
    axes[0].axhline(RISK_EPS, color="k", ls="--", lw=1); axes[0].set_xlabel("acceptance (all held-out images)")
    axes[0].set_ylabel("wrong-remedy risk"); axes[0].set_title(PROPOSED, fontsize=10)
    axes[0].legend(fontsize=6, loc="upper left")
    # selective wrong-remedy curves (seed 0, pooled over held-out sets with equal weight)
    for m, score, ls in [(PROPOSED, "Remedy+kNN", "-"), (PROPOSED, "Remedy", "--"), (PROPOSED, "MSP", ":"), (BASE_HEAD, "MSP", "-.")]:
        curves = []
        for h in dss:
            r = load_any(m, "lodo", h, HEAD_SEEDS[0])
            if r is None:
                continue
            dv = decision_view(r, "lodo")
            if score not in dv["U"]:
                continue
            ut = dv["U"][score][1]; o = np.argsort(ut, kind="stable")
            cov = np.linspace(0.02, 1, 50)
            curves.append([dv["wrong_t"][o][:max(1, int(c * len(o)))].mean() for c in cov])
        if curves:
            axes[1].plot(np.linspace(0.02, 1, 50), np.mean(curves, 0), ls=ls, label=f"{m} / {score}")
    axes[1].set_xlabel("acceptance"); axes[1].set_ylabel("wrong remedies among issued"); axes[1].legend(fontsize=7)
    axes[1].set_title("selective risk curves (seed 0, mean over held-out sets)", fontsize=10)
    plt.tight_layout(); plt.savefig(FIGS / "rq3_gate.png", dpi=120); plt.show()

In [ ]:
# single-source reference splits (if configured): accuracy and gate (only val-CRC is possible with ONE training source,
# because leave-one-source-out calibration needs >= 2 training sources)
single_keys = [k for k in SPLITS if k[0] == "single"]
if single_keys:
    acc_rows, sg_rows = [], []
    for (_, name_) in single_keys:
        for m in GATE_MODELS:
            for s in INNER_SEEDS:
                r = load_any(m, "single", name_, s)
                if r is None:
                    continue
                ev = eval_test(r, "single", name_)
                acc_rows.append(dict(split=name_, model=m, seed=s, macro_f1=ev["macro_f1"], acc=ev["acc"]))
                sg_rows += gate_rows(m, "single", name_, s)[0]
    if acc_rows:
        report("Single-source reference splits - accuracy (seen classes)",
               md_table(pd.DataFrame(acc_rows).groupby(["split", "model"]).agg(macro_f1=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"), acc=("acc", "mean"), seeds=("seed", "size")).reset_index()))
    if sg_rows:
        sg = pd.DataFrame(sg_rows)
        sg.to_csv(TABLES / "gate_single_source.csv", index=False)
        report(f"Single-source reference splits - gate (target risk {RISK_EPS:.0%})",
               md_table(sg.groupby(["held_out", "model", "score", "calibration"])[["accept", "risk", "wrong_among_issued",
                                                                                   "withheld_unseen"]].mean(), "{:.3f}"))

### 12.1 What the app would say (worked cases, proposed model, `Remedy+kNN` / `LOSO-CRC`)
Action texts are **templates that agronomists must verify** before any field use. They are not advice.

In [ ]:
cases = []
for h in dss:
    r = load_any(PROPOSED, "lodo", h, HEAD_SEEDS[0]); lv = calib_views(PROPOSED, "lodo", h, HEAD_SEEDS[0])
    if r is None or not lv:
        continue
    dv = decision_view(r, "lodo")
    lam = crc_lambda([v["U"]["Remedy+kNN"][1] for v in lv], [v["wrong_t"] for v in lv])
    ut = dv["U"]["Remedy+kNN"][1]; acc = ut <= lam
    kinds = {"issued & right": acc & ~dv["wrong_t"], "withheld (would have been wrong)": ~acc & dv["wrong_t"],
             "withheld unseen disease": ~acc & dv["unseen"], "issued & WRONG (failure)": acc & dv["wrong_t"]}
    for kind, msk in kinds.items():
        w_ = np.where(msk)[0]
        if len(w_):
            i = w_[0]
            cases.append(dict(held_out=h, case=kind, true=CLASSES[dv["y"][i]] if dv["y"][i] >= 0 else "?",
                              top_disease=CLASSES[dv["classes"][dv["pt"][i].argmax()]], issued_group=GROUPS[dv["issued_t"][i]],
                              decision="ISSUE" if acc[i] else "WITHHOLD", u_risk=float(ut[i]), lam=lam,
                              message=ACTION[GROUPS[dv["issued_t"][i]]] if acc[i] else
                              "Uncertain: re-take a close, well-lit photo of the affected leaf, or consult the extension officer.",
                              file=idx.rel[int(r["test_idx"][i])]))
if cases:
    cs = pd.DataFrame(cases); cs.to_csv(TABLES / "gate_case_studies.csv", index=False)
    report("Worked cases (UNVERIFIED action templates)", md_table(cs.drop(columns=["message", "file"])))

## 13. Explanations that are tested (RQ4)
SERA's attention over the 14 views is a built-in explanation: *which part of the photo, at which zoom, carried the decision*. Instead of judging heat maps by eye, we test it:
- **Bag deletion faithfulness**: remove the *k* most-attended views (k = 1…4) and measure the drop in the predicted class probability, against removing *k* random views (5 draws). Faithful attention → larger drop.
- **Keep-only-top-k**: keep only the *k* most-attended views. A faithful explanation should keep most of the probability.
- **Correct vs wrong** cross-source predictions: attention entropy, share on the global view, share on the finest tiles (Mann–Whitney U).
- **Lesion pointing** (if `lesion_masks/` from the v1 annotation pack is attached): the lesion fraction in the most-attended tile divided by the mean lesion fraction over tiles (lift > 1 = attention points at lesions), and the hit rate.
- Comparison: SERA vs MIL without erasure (same seed).

In [ ]:
def find_mask_dir(name):
    if not INPUT_ROOT.exists():
        return None
    for p in INPUT_ROOT.rglob(name):
        if p.is_dir():
            return p
    return None
MASKS_A = find_mask_dir("lesion_masks")

def mask_tile_frac(i):
    if MASKS_A is None:
        return None
    for ext_ in (".png", ".jpg"):
        f = MASKS_A / f"{uid_of(i)}{ext_}"
        if f.exists():
            m = np.asarray(Image.open(f).convert("L").resize((TILE_CANVAS, TILE_CANVAS), Image.NEAREST)) > 127
            return np.array([m[y0:y1, x0:x1].mean() for (x0, y0, x1, y1) in TILE_BOXES]) if m.any() else None
    return None

XAI_ROWS, DEL_ROWS, ATT_EXAMPLES = [], [], {}
rng_x = np.random.default_rng(0)
for m in [PROPOSED, M_("MIL", MAIN_FM)]:
    for h in dss:
        r = build_and_train(m, "lodo", h, HEAD_SEEDS[0], keep_X=True)
        S = SPLITS[("lodo", h)]
        pos = {g: i for i, g in enumerate(r["test_idx"])}
        sel = np.array([pos[g] for g in S["test_id"]], dtype=int)
        X = r["X"][2][torch.as_tensor(sel, device=DEVICE)]
        model, classes = r["model"], r["classes"]
        logits, att, _ = head_forward(model, X)
        p0 = softmax(logits); pred = p0.argmax(1); pp = p0[np.arange(len(pred)), pred]
        y = np.searchsorted(classes, Y_ALL[S["test_id"]].numpy())
        Tn = X.shape[1]
        order = np.argsort(-att, axis=1)
        for k in range(1, 5):
            for mode in ("top", "random", "keep-top"):
                draws = 1 if mode != "random" else 5
                drops = []
                for _ in range(draws):
                    keep = np.ones((len(pred), Tn), bool)
                    if mode == "top":
                        np.put_along_axis(keep, order[:, :k], False, 1)
                    elif mode == "random":
                        rnd = np.argsort(rng_x.random((len(pred), Tn)), 1)[:, :k]
                        np.put_along_axis(keep, rnd, False, 1)
                    else:
                        keep[:] = False; np.put_along_axis(keep, order[:, :k], True, 1)
                    lk = head_forward(model, X, keep=torch.as_tensor(keep, device=DEVICE))[0]
                    drops.append(pp - softmax(lk)[np.arange(len(pred)), pred])
                DEL_ROWS.append(dict(model=m, held_out=h, k=k, mode=mode, prob_drop=float(np.mean(drops))))
        ent = -(att * np.log(att + 1e-12)).sum(1) / np.log(Tn)
        fine = att[:, TILE_SCALE[:Tn] == max(TILE_GRIDS)].sum(1)
        for j, i in enumerate(S["test_id"]):
            row = dict(model=m, held_out=h, i=int(i), correct=bool(pred[j] == y[j]), entropy=ent[j],
                       global_share=att[j, 0], finest_share=fine[j], p_pred=pp[j])
            fr = mask_tile_frac(int(i)) if j < XAI_SAMPLES else None
            if fr is not None:
                t_ = int(np.argmax(att[j, 1:])) + 1
                row["lesion_lift"] = fr[t_] / (fr[1:].mean() + 1e-8)
                row["lesion_hit"] = float(fr[t_] >= fr[1:].max() - 1e-9)
            XAI_ROWS.append(row)
        if m == PROPOSED:
            ATT_EXAMPLES[h] = (S["test_id"], att, pred, y, classes)
        del r, X; torch.cuda.empty_cache()

dl = pd.DataFrame(DEL_ROWS)
if len(dl):
    dl.to_csv(TABLES / "xai_bag_deletion.csv", index=False)
    dpv = dl.groupby(["model", "mode", "k"]).prob_drop.mean().unstack("k")
    report("RQ4 - probability drop when views are removed (top-attended vs random; keep-top = only top-k kept)", md_table(dpv))
xa = pd.DataFrame(XAI_ROWS)
if len(xa):
    xa.to_csv(TABLES / "xai_attention_per_image.csv", index=False)
    cols = [c for c in ["entropy", "global_share", "finest_share", "lesion_lift", "lesion_hit"] if c in xa]
    report("RQ4 - attention statistics (held-out ID images, seed 0)", md_table(xa.groupby(["model", "correct"])[cols].mean()))
    fa = []
    for m, g_ in xa.groupby("model"):
        if g_.correct.nunique() < 2:
            continue
        for metric in ["entropy", "global_share", "finest_share"]:
            a_, b_ = g_[g_.correct][metric], g_[~g_.correct][metric]
            fa.append(dict(model=m, metric=metric, correct_mean=a_.mean(), wrong_mean=b_.mean(),
                           mannwhitney_p=stats.mannwhitneyu(a_, b_).pvalue))
    if fa:
        fa = pd.DataFrame(fa); fa.to_csv(TABLES / "xai_correct_vs_wrong.csv", index=False)
        report("RQ4 - attention of correct vs wrong cross-source predictions", md_table(fa, "{:.4f}"))

In [ ]:
# figure: where SERA looks (sum of tile attention over the canvas, global-view share in the title)
show = []
for h, (tid, att, pred, y, classes) in ATT_EXAMPLES.items():
    ok, bad = np.where(pred == y)[0], np.where(pred != y)[0]
    show += [(h, tid, att, pred, y, classes, j) for j in list(ok[:1]) + list(bad[:1])]
if show:
    fig, axes = plt.subplots(2, len(show), figsize=(2.3 * len(show), 4.8), squeeze=False)
    for c_, (h, tid, att, pred, y, classes, j) in enumerate(show):
        i = int(tid[j]); img = CACHE[i].transpose(1, 2, 0)
        heat = np.zeros((TILE_CANVAS, TILE_CANVAS))
        for t_, (x0, y0, x1, y1) in enumerate(TILE_BOXES[:att.shape[1]]):
            if t_ > 0:
                heat[y0:y1, x0:x1] += att[j, t_]
        heat = np.asarray(Image.fromarray((255 * heat / (heat.max() + 1e-8)).astype(np.uint8)).resize((CACHE_SIZE, CACHE_SIZE)))
        axes[0, c_].imshow(img); axes[0, c_].axis("off")
        axes[0, c_].set_title(f"{h[:12]}\ntrue {CLASSES[classes[y[j]]][:12]}", fontsize=7)
        axes[1, c_].imshow(img); axes[1, c_].imshow(heat, cmap="jet", alpha=0.45); axes[1, c_].axis("off")
        axes[1, c_].set_title(f"pred {CLASSES[classes[pred[j]]][:12]}\nglobal view {att[j, 0]:.2f}", fontsize=7)
    plt.tight_layout(); plt.savefig(FIGS / "xai_sera_attention.png", dpi=120); plt.show()

## 14. Efficiency (on-device readiness)
Encoder cost per view (GFLOPs, params) and CPU latency (4 threads, a rough stand-in for a mid-range phone) for 1, 5 and 14 views. The heads are negligible (< 0.3 M parameters). ONNX export of the phone-sized encoder for later TFLite conversion.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
eff = []
for fm in dict.fromkeys([EDGE_FM, MAIN_FM]):
    try:
        enc, mean, std, counted = load_encoder(FM_ENCODERS[fm]); enc = enc.cpu().eval()
    except Exception as e:
        print(fm, "unavailable:", e); continue
    x1 = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)
    try:
        with FlopCounterMode(display=False) as fc, torch.no_grad():
            enc(x1)
        gfl = fc.get_total_flops() / 2e9
    except Exception:
        gfl = np.nan
    torch.set_num_threads(4)
    for nt in (1, 5, N_TILES):
        x = torch.rand(nt, 3, IMG_SIZE, IMG_SIZE); ts = []
        with torch.no_grad():
            enc(x)
            for _ in range(2 if SMOKE else 7):
                t = time.perf_counter(); enc(x); ts.append(time.perf_counter() - t)
        eff.append(dict(encoder=fm, views=nt, params_M=count_params(counted), GFLOPs=gfl * nt, cpu_ms=1000 * np.median(ts)))
    if fm == EDGE_FM:
        for dyn in (False, True):     # legacy tracer first; DINOv3's attention may need the torch.export path
            try:
                torch.onnx.export(enc, (x1,), str(RES / f"{fm}_encoder.onnx"), input_names=["image_normalized"],
                                  output_names=["embedding"], opset_version=17 if not dyn else 18, dynamo=dyn)
                print("ONNX exported:", RES / f"{fm}_encoder.onnx", "| dynamo =", dyn); break
            except Exception as e:
                print(f"ONNX export (dynamo={dyn}) failed:", str(e)[:160])
    del enc; gc.collect()
if eff:
    eff = pd.DataFrame(eff); eff.to_csv(TABLES / "efficiency.csv", index=False)
    report("Efficiency (encoder; 224x224 views; CPU 4 threads)", md_table(eff, "{:.2f}"))

## 15. Blinded expert label audit (RQ1)
D3 can only *flag* classes whose same-named labels look different across sources. Whether that is a labelling error, a different disease stage, or a genuinely different pathogen needs a plant pathologist. This cell exports `expert_audit_pack/` with `AUDIT_PER_CELL` images per (source, class) under **random ids, without source or label**, and a sheet to fill.

Upload the filled sheet as `audit_sheet_filled.csv` (any dataset, attached as input) and re-run. The agreement analysis below then fills itself in: per source and class, the share of images where the expert agrees with the dataset label, plus Cohen's κ per source.

In [ ]:
pack = OUT / "expert_audit_pack"
(pack / "images").mkdir(parents=True, exist_ok=True)
sel_rows = []
for (d, c), g_ in meta.groupby(["dataset", "label_name"]):
    sel_rows += list(g_.sample(min(AUDIT_PER_CELL, len(g_)), random_state=7).index)
order_ = np.random.default_rng(7).permutation(len(sel_rows))
key_rows, sheet_rows = [], []
for n_, j in enumerate(order_):
    i = int(sel_rows[j]); src_f = Path(idx.path[i]); aid = f"A{n_:04d}"
    shutil.copy2(src_f, pack / "images" / f"{aid}{src_f.suffix.lower()}")
    sheet_rows.append(dict(audit_id=aid, file=f"{aid}{src_f.suffix.lower()}", expert_label="", expert_confidence_1to3="", notes=""))
    key_rows.append(dict(audit_id=aid, dataset=idx.dataset[i], source_class=idx.source_class[i], label_name=idx.label_name[i], rel=idx.rel[i]))
pd.DataFrame(sheet_rows).to_csv(pack / "audit_sheet.csv", index=False)
pd.DataFrame(key_rows).to_csv(OUT / "expert_audit_KEY_do_not_share.csv", index=False)
(pack / "README.txt").write_text(
    "For each image, write the disease you see in expert_label, using one of:\n  " + ", ".join(sorted(REMEDY_GROUP)) +
    ", other, unsure\nConfidence: 1 = guess, 2 = probable, 3 = certain. Use notes for co-infection or stage.\n"
    "Do not look up where the images come from. Return the filled sheet as audit_sheet_filled.csv.\n")
print("audit pack:", pack, "|", len(sheet_rows), "images")

filled = next(iter(INPUT_ROOT.rglob("audit_sheet_filled.csv")), None) if INPUT_ROOT.exists() else None
if filled is not None:
    from sklearn.metrics import cohen_kappa_score
    au = pd.read_csv(filled).merge(pd.DataFrame(key_rows), on="audit_id")
    au["expert_label"] = au.expert_label.astype(str).str.strip().str.lower().str.replace(" ", "_")
    au = au[au.expert_label.ne("") & au.expert_label.ne("nan")]
    if MERGE_VIRAL:
        au["expert_label"] = au.expert_label.replace({"iris_yellow_virus": "viral", "virosis": "viral"})
    au["agree"] = au.expert_label == au.label_name
    agr = au.groupby(["label_name", "dataset"]).agg(n=("agree", "size"), agreement=("agree", "mean"),
                                                    expert_unsure=("expert_label", lambda s: s.isin(["unsure", "other"]).mean()))
    agr.to_csv(TABLES / "expert_audit_agreement.csv")
    kap = {d: cohen_kappa_score(g_.label_name, g_.expert_label) for d, g_ in au.groupby("dataset") if g_.label_name.nunique() > 1}
    report("Expert audit - agreement between dataset labels and the blinded expert",
           md_table(agr, "{:.2f}") + "\n\nCohen's kappa per source: " + ", ".join(f"{d}: {k:.2f}" for d, k in kap.items()))
else:
    print("no audit_sheet_filled.csv attached yet")

## 16. External prospective test (only if `onion_external/` is attached)
Models trained on **all four** public sources and gated with LOSO-CRC, calibrated on the four LODO models' out-of-source predictions. Nothing is tuned on the external images.

In [ ]:
if EXT is not None and ("ext", "all") in SPLITS:
    er_rows, eg_rows = [], []
    for m in GATE_MODELS:
        for s in INNER_SEEDS:
            r = load_any(m, "ext", "all", s)
            if r is None:
                continue
            ev = eval_test(r, "ext", "all")
            er_rows.append(dict(model=m, seed=s, macro_f1=ev["macro_f1"], acc=ev["acc"], **{f"F1_{c}": v for c, v in ev["per_class"].items()}))
            a, _ = gate_rows(m, "ext", "all", s); eg_rows += a
    if er_rows:
        e1 = pd.DataFrame(er_rows).groupby("model").mean(numeric_only=True).drop(columns="seed")
        e1.to_csv(TABLES / "external_accuracy.csv"); report("External test - accuracy (classes seen in training)", md_table(e1))
    if eg_rows:
        e2 = (pd.DataFrame(eg_rows).groupby(["model", "score", "calibration"])
              [["accept", "risk", "violation", "wrong_among_issued", "withheld_unseen"]].mean())
        e2.to_csv(TABLES / "external_gate.csv"); report("External test - gate", md_table(e2, "{:.3f}"))
else:
    print("no external set attached (optional)")

## 17. Report and archive

In [ ]:
lines = [f"# {CROP.capitalize()} cross-source diagnosis (SERA pipeline): results report", "",
         f"Generated {time.strftime('%Y-%m-%d %H:%M')} | {'SMOKE TEST' if SMOKE else 'full run'} | classes: {', '.join(CLASSES)} | "
         f"main encoder {MAIN_FM} | proposed {PROPOSED} | head runs {len(list(HEADS.glob('*.json')))} | elapsed {hours():.2f} h", "",
         "Figures: " + ", ".join(sorted(p.name for p in FIGS.glob('*.png'))), ""]
for title, text in REPORT:
    lines += [f"## {title}", "", text, ""]
(RES / "REPORT.md").write_text("\n".join(lines), encoding="utf-8")
save_json(dict(SMOKE=SMOKE, MERGE_VIRAL=MERGE_VIRAL, IMG_SIZE=IMG_SIZE, TILE_GRIDS=TILE_GRIDS, TILE_CANVAS=TILE_CANVAS,
               FM_ENCODERS=FM_ENCODERS, MAIN_FM=MAIN_FM, EDGE_FM=EDGE_FM, HEAD_SEEDS=HEAD_SEEDS, INNER_SEEDS=INNER_SEEDS,
               HEAD_EPOCHS=HEAD_EPOCHS, TILE_DROP=TILE_DROP, KNN_K=KNN_K, RISK_EPS=RISK_EPS, ALPHA=ALPHA,
               classes=CLASSES, torch=torch.__version__, timm=timm.__version__, HEAD_TAG=HEAD_TAG, RUN_TAG=RUN_TAG),
          RES / "config.json")
with zipfile.ZipFile(OUT / "results_light.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in RES.rglob("*"):
        if f.is_file() and f.suffix not in {".pt", ".npz", ".onnx", ".npy"}:
            zf.write(f, f.relative_to(OUT))
print("REPORT:", RES / "REPORT.md", "| archive:", OUT / "results_light.zip", f"| total time {hours():.2f} h")